# Deney 06 — Render Pipeline (Notebook 01+02+03 çıktılarını birleştirir)

**Amaç:** Onaylanmış an + script + seslendirmeyi kullanarak dikey (9:16) bir YouTube Shorts videosu üretmek: kaynak aralığı kes, dikey kadraj uygula, seslendirmeyi orijinal sesle miksle (ducking), altyazı yak, kaliteli kodlama ile dışa aktar.

**Çalıştırma sırası:** Notebook 01 → an onayı → Notebook 02 → script onayı → Notebook 03 → seslendirme onayı → **bu notebook**. Alternatif olarak `00_youtube_automation_master.ipynb` tek başına tüm fazları çalıştırabilir.

**Bu notebook hiçbir yere yükleme/yayınlama yapmaz.** Çıktı sadece `results/render/final_short.mp4` olarak yerelde/Colab oturumunda kalır.

**Tek kaynak politikası (düzeltildi):** Bu notebook artık ffmpeg filtergraph'larını, altyazı üretimini, doğrulamayı ve onay kapısı mantığını **kendi içinde tekrar tanımlamaz**. Hepsi `local_pipeline/render_utils.py` içindeki kanonik fonksiyonlardan gelir — master notebook (`00_...`) ve `verify_local.py` de **aynı** fonksiyonları çağırır, yani üç ayrı kopya yoktur. Modül aşağıdaki bootstrap hücresiyle yüklenir (repo → Drive → `PROJECT_REPO_URL` → gömülü hash-doğrulamalı yedek).

**Onay politikası (üretimden sonra, içeriğe bağlı):** `run_render_pipeline()` **onay argümanı almaz** ve asla `approved` üretemez — yeni render her zaman `review_required` (teknik doğrulama başarısızsa `failed`) olur. Onay, videoyu **izledikten sonra** bölüm 5'teki ayrı hücrede, dosyanın SHA-256'sı doğrulanarak verilir. Yeniden render alırsan hash değişir ve önceki onay otomatik geçersizleşir. `render_validation` bir **teknik** kontrol sonucudur, insan onayı değildir.

In [ ]:
!pip -q install imageio-ffmpeg faster-whisper pandas > /dev/null
print('Kurulum tamam.')

In [ ]:
import os, sys, json
from pathlib import Path

if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            import site
            site.addsitedir(str(venv_site_packages))
            break

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = Path('/content/results') if IS_COLAB else PROJECT_ROOT / 'results'
RENDER_DIR = RESULTS_DIR / 'render'
RENDER_DIR.mkdir(parents=True, exist_ok=True)

STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}
print('Sonuc dizini:', RESULTS_DIR)


## `local_pipeline` kanonik modüllerini yükle (tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['render_utils', 'video_analysis', 'approval']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'render_utils': {
        'sha256': '0b64530651d15439402d8bfc20e22507acc64e8e32650a97cfbd2bd8c0a665fc',
        'encoding': 'zlib+b85',
        'data': (
            'c-rlK*_PWzmf$<TBAh<mAT5x&Xt9ba+9xfgY>j0}M=9IwPUTP_2_|8J01Hc|sBE6oe=sloIrBCB1M>@Wmxw?_0AQwMRo7Hcsmc-v'
            'MBLbJ+_-z}?d`n|%RHFGp?LaqwGJ;t7ACVW>#w7A7)MDc7Gb;&vs~nhAPZ+=vJnRl#$0Hu3Js$5CYgxt)gqcMI#Gh}auFmV{7V^K'
            '2631azIdF*!K5ES#WmJW3NcOB8<EaMv50aJrx($*2bE^xGR&g+W*n!}AReqY9fm8S)jG|JoQtp0SqYt<MOip4;*FT68GHx@Rg!}c'
            '`6^w8q6qV%H|TU;q);=tfDvV(DA#cs%yI$M*5xFQ@&)yHLq!JSSs~H{z-A#9VxaRVrZO{el|=xwW4<n_!9c*97^W{T*XuZf`AW-T'
            'T^3>;6pKM;Z*Q+N&(f6`kLP7kX5n}&M+IP$v<QkQP4Z4hzFP$OB912N=kIx%sOKzHPx+!OqFDbbC+jSohIy{u6bl>!42oNx1w~jy'
            't5CJ8e)|Id{*Wdim%|Q2+p5?r_}S^4zBzvL^Xd4P<L5u0oQjdS?g;p^7rqA!f!awC#CyKjn}wHQoUZZ3cpV4RaFNC^JNPLJFQf1Z'
            '@COhB&}q+S2*Db9eTl!tUX<kJd>&0BtPb4^@Z~&!S*FiWa8^!nqQ*F|4S?7}&wl>y&9gVpPsYFg@a*)}$?Ne?FQ1$|r#^Z`lx#d-'
            'c=-y}1|L@8EGk#{6bIQw*uT8nhsBtr9sqiL{OZlKmoLU|emME*Wc=#&%cswt8)Nlm;XEkg!W+sL^b-1nSTF{}&7ouP@zAfn&Qlmj'
            '60Aaa?>z~B4}K}dX^`ZeTPBY_V41Ivj1K_}^)eh!(>Tq>L7pRwfBxYB{yc?0CWuLpH<Ww&)GfCP-jA=+Y=%YdIq#>i$?^T&1M5R}'
            '5hdeGcz<+cegGT~vnezSU!DeW9va``uqc2+<m+GxxCaGf#V!BiUBG;d)A<<C6L2oSX+DqI=JTf@n}%^*=Hm2m>x3Ts%_sEe;m1zs'
            'o2=YDp92$QkpV(YvuI5l1LkkFMjY+x5>)C*5U$chyn*(EY!fEFc)bY1WRnHvyI-O_%r?3C>ZeT}WKjS+#s>I9TIQiy@K>O<VV35;'
            'c#;JdK+FnQU0?hNeSYiZ;yF-Hc&x#BmSMLD6`=hf-aup6lMpuO1`t4(?zu~hcv8cK)5rbS;YA6R2SqWjqs1aPy8i%c9Nay)cQ2Sc'
            'xPNr?aB^_}{=Mm+g8S3M@WI{L{OC`!d*S_u!Pkcm?|&U09UeVAI1GaM*AE`f=Ci|t2b0<Spa#0kVuZUWigiBR-_L@p!9`Ro$_dO<'
            'HiZ&E)CSXZwSSSO7jd|cmGb>n5GDKRJcb83f_%Dvi_yf<G)w?V%}IFq^!cE8Uo;i_jGb#}WdcVa&}g8_aX?!Pj}UARHctp}6f_B>'
            'f4d<nsA>!!0p0GSBs^b=AHy{qx6GcrheLWAAt$n01A@Z=yPu_TJPD>tk<wloh~u(ISJ38EeE;%|$lyRnCL&3%Bs`eBoLG-l3Va0|'
            'z(rKVVIL1C;;T#wB_w4N!32)GK_e*~!v_Bl#-Dt`Jm%^xwD7^hzdbGd`g7C5!@<!my1+Bi{{=}HtjG;CVSaD+@b2vHz1iXX!Sw5U'
            'hlhW<8{P}1^LrB@3iChRzk3+o4<39yc@P9dH)aP1!Pm3N*GIv<@L+!T-sh$Xj1K!eh=02O-POgS_^10%{<gG0cIuPKfHkPrmHD{M'
            '-`J-uv4`Iu12gyXDR6bKU%ogwmE7HrC%^sr^7Rws?z+`hWQl~i5;&+9bi~m+5I5*h!`Fp6^YF)&JoRLoUWCUP!@$q8>L{e2bPD5B'
            'CQo025y6~i^AK6oPOsDH0H+CTexAm-Ljyk8p%1IDi`+A|IOE?#krx?^!3%&=0VrC%_r)0h(%-zE=zjy^(zF;d%Z-vB0SBRF7DCgj'
            'D2Z|;bFwi_t4JQuq>`FMu}L8b3N;8(_X?VWM%`Xw74Bhg0H33Ew>OB>t1#>KsCqOvAbg=lGt|gneFs@dM2To-)`u2vkp;lcAWu<-'
            'Cs~$eHcNzm=6l~oAhE7OSWP{6IkVTg%`dX;BeB;|c@<txQNp9y0@)8L<sdSJvu<<udRrSxPydpuJ@ZlQgtG`HQa2A6qLzT?lX;>L'
            'xbAAaNt%I-1B8AymPjDi!bAyL2+y8;`~zp)bU~|^Kq-5gy325bWSzup^$ZJe!@hzwGYZ#xXK-`)=LNEVo%*~07}+^zY3uV<lPi2N'
            '_hcW2*AmLM;wSC-YXho;K3Ie#<>?<wpkspR$sGV<WMUrM&K>}gRV}DlikmbF$v!w`N+tH@!~*bxU*i(s6K?bt{5w<7&*k)QC9#ob'
            'Ae|uNc^SuDB(p>}0Ec*%X3+)o9_B>qCI(zbqR|zcAiy_;3DrabHvzf>yw)N>LS9WYqS16wCQHPN2#8NN4px&{Fq9<+$USu*+`oVK'
            'LC+TxKv#Q_xEq6Vjp~aoLF$=HAipod_x#jV1Qm`ERJ&O8bd1MGmmgtnV4->bfRYA)F5_90)rsd%!4fH_rjY<b1eZY+6VFGQp?G6x'
            'vtX3DSVu_$JM8p_<33E<phLjE<3mpHSq5_mJm<4S<iOeh8%fYfb%rXhkWQiSJuq%?B%zWdh{gAGLKCc~co33CN<_JsrB?}Rxlj$2'
            'OyN~hqy*qe28U=Lxm@@pb!9YG;C7kv&E#y-Cse+m60Dq|MhsRqL|t2eeMHqByv$+cr=(ni7T`RI=JPN^6(7#aK+PHbt7dE@%o#t^'
            'xO+~e6c+=<<cGmUSacoUPM;ij9;mlNgP!t!<XVzvAR@z@S&mCvy~@)PICW$z@(t|J72yTygb<cS1?Ukf!@3o2*xv`MCHxH=0LPz?'
            'h;@Y(gQInvF6k#%m;qWw32FQQWCea-4}=N)`*k=)0>T@unvXM!Z#{R<QUm-T%4y2FtX-{dbEB+27Q|@ewtd6I;yGhNaweIMAw6f-'
            'bkh!r@L0g_YQ|xlzeDrmEKLhF>0|EoxHphQ444R>Y55P;RBR)5j3=xDv|r6%GpBm4X3&Ik;}n{OZ#Rdmw;iAk{K%^An`>7cf(|d-'
            'IxgNPyE$hc836Nmb&6NRwaIQ{pJg52!eOg3HPOVcM3Y|C4@L+)Zxv{b7(^SJI3B5<25ANlBYC)X^k>+`@E!PccZ+$0Yt@_WK+(yH'
            'Ehw^$J#sqP>{bTwlg~P9L!w}H&f4^}EJpY4(*X$Rx&{%z;Wex-4#=u7o@|OR@2c{A5E_#t7p#tf8cv0NU+h3Xn>~JACfH$Ski95@'
            'S;E=jV@)1caGYFZC7jbi9*IeS3c)=3;MrZPMw5l4`tLR(bK@M}dY(-g)|l#}gBzxrIN49c(%Y|2_<o8SrW5)HsXO#Z!G>@3Zmlyo'
            'n7ecD=iiQBKO6F#$59?6xmdzZ!|7Hlf|d)AFAQ*7X922Br1*oeLT|mD4EDF7ueCk`x>d_cCt2<i27BpCGypEUz0M7Y)*wv130DEl'
            'Kna}ShbW5!Sc+g4iSD(%&1i+rw>_8*oP|MyXtwt>$d++L-~y3{IiRb!2&cdqZGv?aBiw-_iz#7u+Ud0Jw8rCd*IsKF`rI={ic*H!'
            'WJ`FH0jG}^QtYf+a-&hT<e&Ex=7+7Lc$m-8C_J73xGW0wx|8+*ThB$eDgv_B99C%(;*Z|I`~?EsModCtwKT()!YIXm{`-IMRIh@I'
            'FiK^MY<<la(V7&s@+Ea+<8y5W?nB1~A}+Lm9B4=+q{Gs%U`9?=kCMCqw%qC$Aj33Wt*ANd+(0}g3vMn}B?vlY9x7%8D&SfKu-MWd'
            'J%^IHAZ0PIy}*DV&o17{*80)Ap2aQ0Ay%%?5KE|T_knHjT9+b^HG57vvZ9)9l<oOIrtSc9OrJpF0_n=GiWE+*W%xdnThr`A9Tt7F'
            'QLSaEm28!RSj8L$u+ACo7?kYlLL`jxgv!FIk@cKciP$xi48zqVoS~fvS1badr@{b+d65O{1;SIa*##Ot(7N}0^k;<i4WQuTGy$qt'
            '@G9kTKxTeeENJE`$Yw&lLavVcpA*19{0cNTy#gVMWF=NP3)FxoqWgG=<9Yn{?Xahp%+R(l6Q~YA@9kSWO7<&h1<sc)sXxRw;*grg'
            'UuTEs$S`_CL+K@f(&vM7aYq2Kp$Grco1@{mxjOhxF2$sbVxsp&1cK<;n6)lSkwgc4*eSDOF+{P@7mF~uSQPTdBrdaY9$l;gdI3Ga'
            'v(;HbMF5%?XPyNY=l~$PunlAOK-e#U62`c$lMH1~SQgf_5y%v}i@S5W2!MLvmO(cMfU|=2IXf6$p0fZO2sJM8>-!&+g9Ql4AxhC_'
            'RAF|`LMIQ_gVPcX2GcBEd!+ut0cJp{5i2e-q(h!9rJ~9S#cf;=vvy4evJbKO9(5EDDGo>1G{4*7HP7vK2uL>#$7vQ_03C~gsa+HM'
            'j)Zy?B~w)Ag#Mlb5Sy)mQls2+v}I<u8^(Cc-ml{bxT&+r#W>HV=Vx>N_mSH(Rp75aTK@vaG9rW=U02Atg&s6?Yu?nY{?X13%|h-F'
            'RBjw7&cZojXdG-t-Cz4xz5SzM_lN!h{%u=~n&x@FH>niOQn$hwdX4{lk!?kH5-x!}gvn4CXvaI!Ud-NJ&)V!kIYV-9?DR#D8>7X~'
            'LorFy7zE66x`b^w20j-~otb>y$S+qaH>kst1uQDClSn4eD0m6X3Tn#bJ2;@JP-1`hRO+#c-XpyLybR7oSB*wY<7h2tNZf1bt3^tu'
            'DNq5}2ldhSSh_enBv`tmJtiw>0Z@|jfk6O}%V?NGcMk!6AVePS=-QotEd$n2-MPQTmaKpqGmgj)PYu|*yEpKrxtjbCsO_I1KQjx@'
            '@d1{haa&*oT@e2w;*HjHU?B4m`kv+J{53il9NZuBR=ch+X*&#x0x)_6-++3<;6`^34$iOu?5BEz@`LJ80Dl1~0rH=Zj)t?6M>>Kd'
            '8g;(I8SEUG`Z4*&MD*zhpc~6!>{WO2KWgGXVbZPXqj~?MgV8p_GLfvF!J5kxKgcffAqtnp8T<y>`70k;+Y4GP+x((9khDM%Qg&yY'
            'hf}$u#lJB#Kcatle{1K%!DB!H;F-eND}GV!`w}+2-0`#>`y0_;0OkN`OA;W6{TQ{`mtibIB$~kMV5bM-HB!^ulA<b`11hyqr(dFr'
            'YF+?g%|y3=nV1I9haR#SNt*OOgjtHZ*g~q_e_a3}{_f?QAH+K~3-3f1NK!h<!wjfBLmy0m;1(d~6R-BJJ|*v<kIcke0JBKUF>&I+'
            '-&`f)UB3bjvmOR3<oHo_z-%Y%msvRL1DrIO<^4SE&jVoKbBUO1GA4@PB0%|thPq4;uH<}FM+V{zvP=Lv0KqXSQ5IUko`N|+QtC^Y'
            's36A|5D-@|HAA%&AQ%-eyNmP+<Q@=hB<zSNAo;r%I4<H<p#wApOnD{y6($!jQ_wi~Z4%DW8-~{<j@5j_Y(fP%ZcxpMA2mgn<_@41'
            '>@3Sd5sVww5QL`-G{}LZ2Z)hv)JQ@&ww44Rsuv+Nx(F`Aob4lMeaU4%YP_9V5Nw687!VDoroOcJrQz+LN8+F&-P0ACP0vj6$m^5O'
            'sfpV3TVXf;AOwepuMoq}JC!~touVXB?Iu8E`&fVmeTDF2UIJ-pKp|jc`^%7`2ayb>c;uDEy#EjyrN;A-C&d(7AB+tTm?k1}qITn8'
            'c#}G3?kO={DF;C=gUu$c9dM+zp?ADh&-y_14~FNaE=v+*EF}#ve`t=)zI%E<VgdAFA0AS@B*krz+bcqg4=oeX>()_$ySzpXrM+mc'
            'NNL%zmJG&=no6_OvsWj+Bw4lhmK>-|NjI(}FlOZ(-MvPKJ?Wz+Ng~<=Vc`-o2$Dq-_bPjc4<$Kv$Ztdxg-!SIMuH1h>tch=6ez(<'
            'nGTQ?P!ed%3hIPOT3#$j(vDzz;#S9rCIT{p*5VYECQ|bIQqBv}L?F5#$)#5VX;x#RK8D^H2Nz<(c}LpO=3-2vGL@1w<+2HA5Ueo-'
            'p-YP<aZsvqrE^59f^>+zP13)BVEo<llY@gptC4k(=dxEc%^6uarT_il4Lx<yb0{oE+-Km$$s5Qt402WISCAD_Q)3MWRU{Oe5hX$o'
            '0jAFx@FE0e33~vo0h^U|71pC&9(V6JnvY?@&^a-j0oKqt?rMG|n7X=GesHwB%MkEl?2Aj3dZ2BJ4pHttYZwr*E?sZr)mnn8%CD#E'
            '5vv1~Wn9uG)jwCc*PwbvcakV6!<+33s|e(&TyJOUW~2JP65HCFtKNL6&QV1ETzD_`k3I7YoJScP-6SVk(7qJ!jG?{5t;RiMUI6Dp'
            '0-lBW8qYX{hPv-Zbci~@U<z}<^eo#@xf~7XWfB!qNg86$r3WcI;u9I+2exkob9F1&P(<x~HEH|ea13-B{~H|GlcrEha!Mqic4uT('
            'KkWH}e@kfi0y{PG^2FOIbglqma1oB?o{UYpMv%k7!F;=aP0;9B<cV0Ovlw<+?Es<EYb3YHRS>BSC0F@7VC$>BU7f&GoPFZZ!KBwt'
            'R}9F?4t2LiJ`B44h6|m%Ut|1B6Raf=OlMcsy01@Q=`4{Tad!L9RADfU0iBF`ioX1w;tI<?-&ZRtPl}O6Fj4o2w7_^pc32>lmG52)'
            'v+z-jg2N*;vj>gHW+2x`Vp|={D%9GIW|BwQA@+3X)UQT`hIe(^D|t;R(pfafyM|(2N^M$$?znHV1_kX7aG<}F^#t<JR|=Z^HOKS2'
            'cqjhH>B|>%esDY*LTZq!a6KLS#5{=ejgT>R#BsAz(0~5>f5?!xoHVv9k_G4trQPPyOrne=yzqS#r)Sz?0$nWQ8M6DforQ4=gdP-?'
            '(m6FmK1LV-s3QMdjg2{dyO2;>IDm_ZOExkNDg@ibfS(ynAlrCp^75WF{nZOz0s5WJ@gI!lBV3U7g7XSFZ(!_i!vc0Jo$zjUzGa8X'
            '*-=~6W!aCcJt?;W4r)Mwm0eXB2B9#WXYDcylqpJKMqN?SjPl=s+}aQ4bF_^SjtJSwO@sAKuo38;+-^SMy)+WCY)vKx6dBba%Vs3#'
            'pkqGk5vz;##1Ptn6G7<dA&aCGZyVYso*F7jF*%5@f=x~;h`iEB;3g`h?5>Q+1dt+Ck!40P_2nCZNJYAUoVFj}$UwYA)u)#D;kit7'
            '=m|O6b(566(w|7FW#}>ch=6fStdfv1yV%FL$2gJ8acNyGtp-K{ub~rhL>`d9+pS<N^o$URv3)SEJ+&Q%;_TW(6r7HW&6=IRKzK{p'
            '0e(gn>?kSzO?u?9{@-}F^~GQ?P+P^5n+j0VQz9ALxVi9_ioHQ<fdR%S@2@ko|A%alg+6Ixuv~lU#6GvHC@bPa7G8w!`LIO?fkn9r'
            'k}AZ9jd_S8&`uS-s)LdjLEfiv?&A}wk@q{=*_ga~_tgNHv6jjYKx|-9BlL2WG1tIGyUGlvdU^xhK=k<F)im!>MV@&cW&wLg1gmJ1'
            'X16DardaL@<WUQu`%(p+lM75vj5W(mE09?*_7~3#DWHj;e*8rSThGZgh&)e;+cf|x8?azYuprli1a?TgUIde{$RDBTtBkwv-f6P>'
            '?wv@{R)3|qDkr*&W2MCLi_>R<i0B$9gy@pJ`h9@MzL>5LkM7@NEg^1j7H17$EDMs9Jc~khTVTZ(?OIftV-Pi>6gnFDXgIy36Oy?y'
            'lDiFf*w89Ka{>Tb05%s3Ko~?c5u<91w;dw>K+UwCSisbu-#qO<d}P3r#|K$n5R-#BD%GYznX}+Q8zN(COA>}4pd6n*e)bIdkbx+e'
            '5B^FgnC3~11BeB6DQqis4Ef?y)WjeUgKWC!W);<Yn}5}PJG;{xR)2TD9ljmluWx$>TZSNM5*y*2GP(Q`stw`4v;X-vIsfY0U$Fd;'
            'e>kAnFn4=U$7E%yfvW6B)?k$alRV9hPMhz|9?18%<Ii2vkmj3U6Nz+LZisjoEc)Sn#SX|NWA>BuG_vPL_e`r--Z(eiZ~y$o+gT4*'
            '>H5gOzwN!(Pu*|v1I&y8Z-6$@b@#w<#Un}%K+(5OHzLa&_IfO{sJ2O%!Fkcu<q^<a|LErWasrg9>Bdu!{xy$0Y5UeZoUw?*+qOb='
            'ylv0fCL9u`yIVu53O+FAS&j;Tq%1;sP<n}xQ-kLvH>FIwF{J)DlLW9Qz7lsI92|%{G`kWyzI}l2ST!S?>dDS)XSa%#kWYRaA7FN2'
            'yR~>w7BFgD=MK@UD1pMCaaagyAT>|=3Twbgf+gR<ae>5G@f25RU(!(_es#duu48<{*4@Vjn)8CWo2O%Lox(PfSr4ISq57%pMcQ$T'
            '2&BVjdyQr^rsseKWC>GBSv7{BK*!DkecsH$^t3HV6A5HViI)ukVe;JZbd#(y9K=YL$hj^-YmOv9I2IqZ&GZGB%8QFIv*4S8UZap#'
            ')6HnFwN?RJF0fxsTf64Brs*ZMUqIUJOliBR!={X@XNQ#99ow{Vqcmjp&t6H1&MtR&OO*C2XI0Y)Ousa}!J0L6uxT5u!XQ}=?gPaL'
            '-nV?ljXyYOP-m1RVdNo3HUaCAM8!t&!pim;NUs^3SM=L<B@>di0^SF>LgCDl9xs9L7?#H+p$YV<1~OctI|+Wm9?8Oh+1?Bor6|rB'
            'Az=Sa<C0tw+4WNi$GrEh0_N9;H*CM26sSlfVeHo-RBY%)@nMSmHPa3ja&hwm&xtw`<n{3+mI1p`PQ*=zQ5-2cP^x{awx@HH(=G)}'
            'B~Fxe(@2^s!zyo+{Z`RLN~siz3jL<UFFa*x%Ao+gwIvmve-5p~d^%fczM*NZUV9Q>d1_^J)>e{8I?*W-i1tOt0DR_=hc^4CduSoD'
            '19$30LH;I&x}h+QhhCvYu!|APlt4jzBMv&I+|}5&VHl>*Do*j*PI`jAMa;wSl<O=y{D}{I=aUV=2nwC18S29f7{>NX@@Nu=#y6g0'
            'CBN38Z+rDvFS)uE5V@g-a+H3xXNGE7Whj?LnGu`yDaK5F!-;HBE=dNRX+3j?f$NC!Ma6`0FU}mC!gBJtbE_zb9qrL%)C$&?c6U*0'
            'soRS-EO&e1`tsMyI`hLi%yY@E!3=y}EU07xL;()Esr000=ceRk$PYQGQAm<kZ(;N0T;Je#Dcnvn*5`)g^~s=^dp_^nxUlHYZFRdx'
            'bmGQU#(GvCdlr4MKvu|j<qM>O*4MLxa~=eX6FtA>vy?7+bquI*i@G-VuCKQ5uNOwB{0&&xy|Tg`S^PpWy0G4J4|)xue7C}KOOP>z'
            'nE9%(5KHUZuPYG>%J4@mB<%7SFj`@W1gclt$~&)F`MPfacZj3*4fAoY6{_yLUIde4qCu+{ott;I22Ac6MJ4^aoErKr>B8{rvT8E='
            'mi9To4KTVD*rM)bk9N-`?jCurk@-AV0dRF1!4J145}hwa|MUF;Xs1dUco8O{9C7y+i7XZGv<fh<OXo}NZ03RF^z)1H>ysBxPF{;O'
            '?CKDBy3gBfHVP8@j`NJ21Aq%zt(8dJ=u%xt=CGb!;IRFg9Bl~nS%{f@X5=0MT<|KLsd7Fs=v>$o`}3-#uIz)nu1dynlq+os*<ZU<'
            'sZVe4v4`^g;laUyjV(ZED!st);AplTUYkk*;}t=n-~Kf>I6SzE$vjLq_$eUd7<Kf3%H%?xch!URA3{d+oFx>>NTott?j6yLR%Ntk'
            'QUN$a{X>ueUGb2S@IBONdeSN(x^u#pK8OhU8hQI-1A_8EoT5jyj4MhuLV^?I+ACwjbZ^)heO1Oqv?eX~DnxfRlnnrTQC5KgYEc+}'
            'R5~^~+$hdX=q|gVMDXE^dws6_a!kiRa)QLXL7kocHC7Olq#-<+c0KH?)=lxu4Y>>a;~Xug#O2dVV$LwR683LM9ohDF_k@y-$q=al'
            'xvm=n0-$4q;Ns4x+5z0Odjf*1$zqnt(&vB1G5PkLVHvx?m@2Djm5@5N?=&wfx2`7*9TPgnpz?Bs`442#1IyP-<<Fs^x=vou2{ps`'
            'y;uKCy}!fU1{D!^#GyN>X8SwKl1rF^`c3P~v!yIRn~?*3-<*~PuQG(qf`K`w&<|@dzar@M>Wv6zES1JL-B5*Yp77WyhV#idm{(tO'
            '4WhZ+F62?Oc$@9g)cdC0(%gMXW)ZZclN`tnCrcM!+C)*SJ1ZLj+l4I}6V2ZH2oaGoOkiy@%ED99QGxv&9U5E@0Z_vtJ@D+kp{KV#'
            '&-bnyPd<RHQK7em<Mtb|qaaO8Z@bsh2$f+H*iu4yfL59Y8UBy8y58GF&NNl()y-MT;X!3(orDw+tK(Cv;3Fv%uue|tXq?P###uF9'
            'V?<aPR%g}?QCmWuMPm#wi%Ex}0jqq-agbb;!G-)_g>~EnJxZbiQw(szStHq;K0Dv&Dab|Bs99&{9GpaZ`g&FQ<cy<>1Y<eic%-Ag'
            'GWF8F5ib)uU8Q4$=t>>5Q2EBE8y*b`6(yzsrSLHk7Sln`3FW49gwlLkOG0O}GKcFExo@b)VB_T?KyS%hdTPoMdpYYlW)<rVtRBTN'
            'R7!BzKGH!XQ+j{7spLq<9Hegq9b10E=$XJXIaV_%1Id5opFbfIRM}AoCt$tIuiZMrU_zS9Xc~^Z>AIx+f~$3jn*qd~kq2|~(6f$Q'
            'Vm!u})yo%S(p`?YsX?_L>~IF83QfoXTZ?k6X-rW0lrDpr40UXXbwECr+>FPR3K{yImN_lpNGdH9#*fs4FD4)-(`ZNcZD=-%kGl46'
            'RP&PGbW`3sH5W)d4fbc?(DH$G3grW0dkjs5Iiu$;vsP#HT3tWvJvK>mrcX=E8jW7RfSI$h9)fWrU)?~T$YNo&*Qfo9<-ObK+}x}v'
            'cJCL_hrmrmBf69dr1!|)o?7$0EMr*jC=RB<a$t<JMd?Te>6R=Q?OKBhjakn*{zye<<2Gn%p{Aj*8?P%uT<r&9@1f<)z1!;5*favD'
            'vEyBvaLSaEl0V0vXI_`{m7_;W5{XT^>=~@HWO`8LJ`xWs@MB44VN+3SXVEYM_OPmP&P)pKoeU$#x(B{yXz{eJO7@JPD@0t%NH}U5'
            '=Dud%jV?G`yp+YP8LPH^1U##5L~s5scKxvDd~4z8>~L5dqhDoFtQaw~hq}tSyRG?T@fx>^*UZ1nHc7CGrYd`4-n|X;asvmUa*uv8'
            'gL7Lp=T>0;04sD`hKNm;EYwx!npk6kaCpZ?a>6BCVL9bGEiLP-Fq%kN5<>=sTs-G0$)&4WpWZK`i$#noSGv~ZrPV+HY!YLGv?B(2'
            'Eg7(;o@9?CkeZTHjAo);0&H{Ne<Rl9za>L*UZdH@E2&B1MRr&aCncc#RF_+o`jNgf&(qU%&;E$+EH~^g%^t}r4Z0yySE51l;PAjX'
            'Z_HVyllEXiDy<JIM?2P_B@?nm9-SFXOm9Q!xo!E|-59sVSVtTp^9?*#RXaW3e1&t@)JLrQ=Mh6{Ze40wsOxtGoZ=7b>G*K=^wKs$'
            '(d*4>U)l4!)l&0&;&Jb`tg@-_lh)|~Q>7FeUEdh7@foKY`4?Z@YXH6N{TFh}^7#KGZrSZ-gQicO+Neg7C%402OZdha?o=h9ZM8|@'
            '{(~$!9xbZ~`0Oyo3#N9n=(F;JIDP#_+{m3v30S2yY*ndPv(?I&G|aw@H}x}S)Nt%{nwSQIcsn%O{@FyCYs<?6R++hjJinzKNGexh'
            '5hKSW<Qv8r+cd<6ncj^iqB?B07S-%U4RtM}k*a_e9bY(|tyefy<tv*Jb#gnrmQ6}45yGR#!L<pLUU}wRa;mqkW>xJuRB?T9Dv+|?'
            'P387~{`dd#jF;E=u4}&8;+vlN<}Ka#Ahc$X{AG0-a@v|KpVW<1^0nlL+mFebLF?|pE<d~@^ZqFJ&m;?j<@h34yRLl??pH>#s-u;u'
            '?0X!S($z`Mf-GA3BF)HkmOWwD3pf<302;;KP6Iq|6bs2t#keCfN+!-)bVCu{Vi`g0I#+RX0019G-jed%*8<iVFg7ct6lT#1GZLbu'
            'n*pH#g2YW#=)AYvS%Ay3CR+66O9|}k;JngiTVP<u9SEve4MzX)T)}rju_b>A-*R*jC@>`!nk&qT(gK*N(0nY6n%HjF8z82Bfq#9A'
            'e+}`kN1l`9mP0njgeUs?6pJW=s4QFTNjp*ZkV%k6>Rz33!PEv=(^@HXi^2L2tNMiK&Gq$Cd?QWa_Z7e7ZieRixCY-SGy0*bYwd{U'
            'ip?|Ma@U8%Q*$2cTpDkH%5K5a*<kgm_D9EgSvL+g?T#G{Nn*$)EEAHvJ&FV(AgWFDi%(!oG8G3KO!Ys0uc*`E!NY@HJio57(KY^b'
            'gkB9Q`>z(Ro3B)?|IycADS%%eA%W!kr*xvrqn*8KAjpk`cLxaf(k(rX#+$bqXtQ+{msPuVA6Na4zE+Eq{wc?*T~+Msw#?I@!HxS@'
            'L=AC#dWs&!C~9()0)-e*>^S2VCih*ZOQll4l+0mFSzidyNq|8$2mt+}!+<h{hJ1)vhXuLnaDphdxbWq(l&H1LL^_w8EGL9Q7$icl'
            '2X=6oMl-Iqr#7!Vvw-YwD&0vq+Y{k@9!@#FJ)KvYnq`m$bjfUR!C7Q4SOp^<2kXaW(WfZcRZ3BOGK7$NVChPDb788MHnzZE1xEVG'
            'K)q2#6ixSJ2J8lV87V2fCz5H$6^mef75DxDIQK-%g4G2w@SAk0BD)rxqsLTqdS|Ds%6*p1({u9zFMf}HaYJ!=Z*XvM$M`z;UcnxJ'
            '9p?Ws6y)Gj5sBWmsoZaFxs6p0S_|TE_8m-3m>nmx(?u{#uZDnG)>W&&X2JRtJ#&WQ$b4@;pZ#+0j-W!hHJ+!WOdpCD0p6I4x2+`l'
            'iS<#hvIt1Z<}u-)FHVUAHD0_d3q(-$vy#%h??6UTufI!U5SO0;yB$q^@iVf)SPS66dA$q)2t49w|3&`$8~n@r(-+4WP-=swDWUP`'
            '9zX?}A1JTHJ|FKY1ZwzPJ-k*Azxd)r+PJNSk@I*=i2W<H;$Lg}y!C0ZG}Z0F3!8Sq3zryfea`^T6cPXU|NO((59MS3&_AI6fm}fU'
            'UGoub)2ZI@VvA+``|#h91r`n-_&>XuN%9Y!J8VUU6QX2|i10kvpoDVDT8|S{a>!|B6xlVQL{NP3Ce)YJ@#!rIr{!|NgaPHD_dfry'
            'b^g53dv=m2S=>0M8!EY1d`RaOo>HS~3&$GGxfwKLh(8na)&!O<-S3Mr4Q_0=LKV(5m^S7Z2-Do02)D`nqub2S9B}9Dta<&Wuon1M'
            '7^tGfZZX42w+{46@q(NJ0HH;3mD<t~9BmY5GElm4B?STQU&2w|01b2RbME4`$mu|fH;;@vn|{ZTJ0n{a_Li}*0T{^T>7_hu=K(T2'
            '>geoqR*C7fT$M4i;CToqJYLj?7g!BC%^FC|G}VA^Qg&j^q+g%HK7Kg-MBQhj=Cyf=O(kr52OVV{N2!e)RO_bfsy6iPwx%|KI$CS('
            '%rqe8u1m4Wn$P3d<XYLf$~l-z8ZokTB)Y}MnXX2GUjFpZJmL0~C`qH0>zsSn*KglW>*vEizkNG7+-|RRcemb+K%7%Yf@rQ+A7wLE'
            'w2w*m%%LBIx~}5E2GLNXeX8brPof}BFVI*Dr<toGBul}z_8!dzTfgZZOXCoi=c|8Rb-MA=%A5S-B(Hc!4y8vI7@#Js?l9MUf1>1v'
            'G0&GbG$}n=7pQOMH`z+dCBhWMpkE+_+C9RJe1v^TqH!Bc!_5|pkG>gkew#}X=QrJlxM;r$abezqcs4wIaPD03B10fp;oM!M#dwvn'
            'wGZ=zsH+h3Fy;Ur{-|G5Eg2F;H-F$klSQ+z$@cv!!f0N3c?#wxLd9ShE?GsWvgP1*r3~*w>yB;(*w?W`AkJJFP5b;ZNP>8iNBI~*'
            'm>>9#ZnAGs;?s^+mxWh+%Ucz1KV77Gn8?~jMAC|qxM81_M=m`pt&*><_9MB6E(lT)_fKKGFVf;E>fd#iwS>%tbD335>b!EkE>Os@'
            'pIMYIMZeEk&sKa?gM2xS!ysekE~QlRoI=Hy4uye?EH<;RF*g}ZvWU`R8}0{M<to<`v&0}OJIEko_V~&ma^sOvy!Jxi*;K*dN_Gjk'
            ';p<D)RJC5J$}$2=Kru}t7eURgj&(WNE@1pEYROWnudLG^-`HlYpabBB#+_oaGUa^5D^65uRUi?S0NyZAznUDN7_~9^Y7Pu)+0=Ev'
            'K5hc%_R@EUC*htCUE5}q(=|*9&>9+vkHPhYRI~PaH(cI4-vfy{iX&9D1j4(fIoW!^y=<a9h+*qpMDb8u6X-9p?ZCw00j8&ebKpa{'
            '6v&VQoNTaXDNMR*b-pp?Z-@S5zGqJ?8aVlB+Y_}#DS#X9UhCgm%o(<U&QP!2p1An(O~X>`?3oLod+oe|_UkLoeXPB&_f>Va{Rkj^'
            '81V@0WqM6_P41$;6@_SkMyZXj)J2h)qQ@pox<mH+Pw9kv_BahUkT%_T{oAK>py`~mX{FJfMp9#*eO}L4ExYiUUGV&qmTWgja{8bb'
            'O_zK!Zr2V_n<e&M)cJv=nHxwjfd=vxX%c<FOBKZu7`BK`+Cn7+1?n)12EvvSydB7n8zWzaNt8qaQ)-J~6M)D)@C<W*l`Yqqpb_Jb'
            'k=dz@#ZWFgxG6Oi$5=M?(unQWYzRekr&;tn-L<d`Hc7CQiQ?R?0L0T+%dI$*t``Q`>T4R8pPPHFVNf<c-fnm`+EY7RV7RqHEyH%N'
            ')4>i!ALN2}o3P_UfF$s`#)eQ4<51t`9^TQoj554SGRk4*oC|9_XC{?dyn(v*YqVKSZ6$5ln_|}tQuyzu&)M9o?~SgxVbvr7?}!_$'
            'ngc+{C3Ez@z5LNdCC|#_^{E`T*;nJn8~c(b?H8mYySqkCvt|j0-;J44QL2A_Z<eRt8cwQq4LukVsAf^7pY^4CqWNMr<E96?9l*%S'
            'q)esii;<d1*+e^*?H|^No0IlEc2Lz5GMA=`>Sn*HzAofiMgMf*n;&oCtgTCbglD#{W4mEfDvl!c>;%c2*p18&s*quiW@}<Y6}oiZ'
            'fvZ0{w&fQ8Z>w}Ql{b`bHv8UCsma!RLlwtDZI`x688+)1VVdaBQusEi0Oz);rljWNHb%D>NR~nQ{_$>OO$ugLefj#?_s?D&KWAq@'
            ';$E~jpqXbY%9VHxBn@wkM8C4Wq6^inquQi9lWH~6LF5DRjMd<?G-Rbcr9TG>`TY4$C<4EEbBd$Fac2W^$ai~Ax7FcAvt_|ILHIKL'
            'bXZ-&!h?u%!@&{joB3>YtN{D%^M&p*>0HOuM^b~_6@9-F!<V_9*|$#W@N1uK+cf7T<M1VUXOew?l1?^4N|y19GLB!#&t1v!ko*5Y'
            'dHgemsH0a(@|uQnB3PZ^Mim?AOm{BT?Y7cmzeekkApfO|vT!U(rQxlwr7yQHJB6&D7%Sm0jaCs|*qm{)bQj82iA<UmhJ{{I{DIwT'
            'se`Z?W-2Uo0ct>#jRJA4xN0hOK2&>!GM=Z(i{G7$GtYSAop0^&s}9O5S)4|9%!$Krk-Q#KDAsaxZNc8wg>!U40M>DnF5MfwVXDJR'
            '+(B3~4l<l#x*4tObgj2_FZXliQqQ@ra%OewykqPpx+z}foU7jSoDzf868`S41N3cCF(IVI8>dU-ZXe!E=*E8$lKe&K<RiKX!23P|'
            '3KRqNo5AiW(HVM^W=j9_i7|?w%w<VxZZn3>D&tnps-`vGr7t!&Asy0Vw9&bKnr7x(Q%T$71mpEy8~QCO&?E;dlDjTEbpg~ZYT<pc'
            'Du`0okcKsj+dx;9GA@kyvp#pNU%owf8yu`Vsm(`?4Nw|4{O;+L%R9I*wXsis{_f4QH_uPTzy9#-^wr7h@lP+GoIF3h^#=Fwwt2d='
            'mhZLI2bMBVQ<Ib`y1g@BWZOrMs6F?pt8mp$Z|Q4q>(#FF{7Z@5nvH8br8Jz@-{Ue8Kd3{02Ean<8?H#e-2qi`lS4DN8e@&qbhB2y'
            'LPA|$<5pvb^!)*sEyJw0FJr^n8Sb%;vv8CvT2OB4CS{PWHO3bNskO$r8FL{DJvP+_gE{aFlt;Rod1pcY!};|Qy3p^K@80lsbFc%%'
            'ZDzq!)~9y0RNKbbt9OSHL|lfOKt#((0RKIYBQb#!Mr{^20PvEbQlu~;@Xb8~yJ?n$%OS>A;gM0m`S1wk`Xoq3*R>X5Te)XpwSJPI'
            'V9X%nr9)+0w5-b4@UmmAgO;KUI#s<xBd6WanWw*c=MA7_6gawT)kLp$aQIdEIcRvM5N7nwmCYKVx`o`nR<fD_*%sg$;o3!9s9KzJ'
            'W}tfC1XF*l%|MlCtPYRgx51;Y4e(ertbvoSo1yW?8nDV~WyI$CMwG0K`aU}^_Z%>`0be`sGy-#c0DSUlwohF`C(LAs8zoHeLS6d='
            'rxM+!BiU32iG4wsnR}HU*izVZtjk(SM1eMa^>dod^)uaVkMs%J$O4SzgPvo&&%xk+1Eu;1fH|p6#Nk8uPv&)yLr@ngU|_@%M`D=D'
            'D)E&Ct0(sN#Sy$bvUp{EtwJ4&{L~JnQ+dA|=3;w-_2<EAea|y_ZcO3W^^`OsC4ps_N3p?XH{PA?o374%x;2)WLd87y%sXyurqt^f'
            'W^u(|n?5T2Rj@`{elWZo9?*3;`b982_}26~?N5hv*;pJ+-XA@<M}>J&@ewn4OH_c88Qn5=#1Plf`|*55UpD3Cy`zJ*XWqhd3R@sd'
            'hawR72Zu^)k%fJEH8tuOj*lP1Y=!Gw<k5TbxKBdNm;?gZ^OvV5rv@}sX~e+>FLYJ6%~Ur>lW!pcW7x=nl6D5~jc1JrT`XsSBZZIm'
            'Gwi#yyb4>kt{T~Up!>7u4a6}VuvtNxo>Ph!Ku>=I9)+F>yoz)lunUXGY5ZDK#93#h!d3U2LKwJ70~+N`2Z`+}k-8NgSB46jD45{g'
            'TvLpnH7>{T`oWae##!0`Uk1TcB8-jcO1>~OXpGo?!}Zi78`-LjqjS>a;m0PXak|6qPC@P{-pMYx=}JY?uDr^eV?bVqo>bjy$O~=R'
            '#p`YEzVS-!cxm)n!Uzgf(E(B{BMxnNB$R2aPw(Sm6MTqJa4AvQ7v;e!_%Kjp+s<q=61SEq9d60g1Zc%2qQFaKr^}IZO>|tf?3W}}'
            'RyenlAgmrIT}$24vBQZf^OqAY-LCbokhdXbdWWo6)zg8~?JT_CX-Rw0#FsjHJkr%-io>1w;m)>NeavqNCw&p25v;l!@=Nircx9id'
            'rDZm36$YeHMJFvjT<~hLh;+sRrXZXySS718&-&_XydaxTu^cZz>Gy#SBTTxmj{UnZ8we9JzVSm+HYkb&xVlGs6(rFd#^KBBx{yjL'
            '`o*jUXJDpmG2}~zy6imI02TDjtG*;WoL8JOa#5p^A<53go}*WO_59~k(oh>Z_&sWwsOBPw=Mo}8g0bM08K@VyM1?ie#v~o{JdKqw'
            'iMg0mIs|!LVPPX^$F?;ES;8EjdAU&??y$j3rK>uU?{}qKPoNshG%52{APTySSpZtZh#QaOavVQ@^V{*iK0AH(uM%?_90*PUAufYV'
            '07-#4{{ZU~O~VX!H_#!`U6miA<$kctfhS2|T^0#2DB4%4SA%xU$SQ?Mb=(!m(d0oQq;mQMgxE9-v$x(X&||=60o@&d2nJ4UOj0}@'
            'Sh?UTCd!?&p0x-$W#nvR&MNZ^6yb*LWTbur6pLK)jlUdQOqn;0N517(5y^&tMs?9|ckW&5dRx8Wb2u!s2>r8cV!G{lJAxx6ovGv4'
            'jykLvQ44_!-`Blh#6hg`)YVstof%(b!Fr){jOf>#cZfcA4STb1G48(2fxBylPU5@QOdTNu6jIqBW}hj_Y1G_$mZ6pp|GJTucD573'
            '-vYZi@VHY;S>rbh&Bll-)^KD>F-D~FW@HH_ey0{oj2eqkUG3plk;bFCCc}oKMB!0Qpf)2u>)HSpiV6d*%lURmSBWudRE^kmYz4)r'
            'LE&MS*Yf$OF4#MnQ!L?o)FQw)X0j?Lys9!x1&<1T;<w22@V@BQF~Lavn357U%3M_yiTf&nqAcSz)ElR5jz&W&14TyObxY4Ql-}0c'
            'qYEl@cg)+>Ya?gsq^QAs?`)m{+Re^!b-e4^)NH+Lx4+vVUgC>$jw;EsOOR>K_v%e>?nrW!P@CS|oU{RTkhFREj^nEIDT`&(e+@=b'
            '5_h^ijKDp`9d}*xblLh=Q0}OTl=rP@+>sG(>e?cV2-?wIRc&<S%V<ucRkeS&AGvsX<jqS6=)}In*aKODer&y4ET2AacP?LbIS{YL'
            'cuLpVzefm<lm*T{jeKWF=SoN-@r<v3Z9Z7I1V!}MbTv{cVp$YKqPh1ZT80>GKg)vO#T|WG7U2?Y?!fYd<auqVx!~BAk^p-hAa{xZ'
            'Hxvt1MGqk5!?Dxr0shs!wJIoTk<{UnJ%Km`>qUe5hgY$t|GBmL$I<aA4G`Vrr1gEh8<jz~P)<Mx)601rT#&SS2USk;QVeRXw)B5k'
            'bs-TaMxVp!Mg~uq3#Qy(X)!wum?WJ?q@n$Bv@8gBl86EUvS8U{bC=Clb<rusxo`2NB*K1uC+y_Q4Cg}U+Mdzjpdp{ri0`y8-9Nt)'
            'N4_|`k6H0LrbW~l%$b@Lnl!#b2!7D4Ch1idA3gvpHz<oKdVHr$O}ee%@TR<yZ{G8yT3b5X%sjT9)~d*hg3je}hnhe5xx0a}w7Z{M'
            'hkz$n6PW5EOJnq+xIk0mBkARQ8ETplB`6w2$uvaC9<$ShOV$CKp(;1^wquD*WYU`jivXn44{@Myi|>3#os#TVX{5qS<5}r-48L$2'
            'WtAw82rIqf)GM?ceuy#~TZ1(^p<#YDm2DJFJVq>K&0Cdg&_zuxK1jTxQfsG5`{txi7vxiC%+pxk{<LEXRZJ!8({R=8Zl95<5H*C%'
            'P^T0V*)YTb=IP2g<Qe6#ba`%^KRkfXgOGGuR;ec=C)u-+pHjwFR5>hry?VAX6W5H8s1^WDkd%=uB7pN4P1a8LHrDQtsX5D4w?2v<'
            '9L74YwfzONYIfXUZ+i;ti?6vf$aJ_u^9V0pkUb;X)Y2!j*MCL})_s~T^R=8cr+<!!{W-_c5|1o*#L&Vo-~Fyg)X?ElyHyAV(*)o#'
            'W+jNfQ#gPohMwuimoI)k{plE;=#_3sQZFPNjyl3qx+cGww1tUH+(fHd%qDCMys^g>HHE{IsT|P1i*!{2O?$Mn8@22qw<fe!cPvG{'
            'Op2|jt~|<D!3=tV9+{*~skm`ty?|tZYzmudMik3V-RwmPoY}bA?!~Q6L;&AK-gRF)8EL)7Z~<Aj@tX^v(I<3h`)m4*@k);89}#EU'
            '7qGFK;YWw<)Lt4F#v0S6$aH<;D!L8d8mPI`i`vP68_#{$+y|mR7xf+}<-3{&uO)88%_Qd;inZ!!YnVCHqK`Kq8b6(qrrHT}ADddJ'
            '#**^#&Y{?$m-0;Bvg&9PREzJdiZ(*sVa`FsxQ(&LhNE4rw1XD)pw(IH7I0X1q)_|vlc31Z_F=0}SQ`Y}IyEkt10o-mCQFsOvn@R-'
            '-uY$zZYit$Ose^7b_%U#mEJ4ns}-J}$2M?sc5e-hsf<1ocYw7@?uIO#cu)(TM;2V|X5<!W16X;V4I7-;c2$!2f0L5et|Y1cE_I`f'
            '$lckr!qhf6DoaBPkeWB!P~1Wl>^1wfnU&kL!d}gD&8*d;*!F7fX>O$k9k=H?q1kn$dcC=dnK0%i?X|8{%Oi86B3tn?=Fvl%o9wcI'
            '5#d7sxQf!edfWbNseho1tqy7|iwdHl=?vC*l2#{T?J%uV%SIwOEki3!58LbPGJI!^QAl3a<20BRCezfQEOsg=jeo6zHU*eo^404q'
            '!o89VT;_zYAH*dg1vVd_=45aQr&_b}6|X^V{NRqquzb-TLl#Dv(a;OKS+(0Xh9)bU(G~Nx+mRNr6ZyK9PY#4uDG?E%?OTmORIUJ('
            'x6RP;VCmTD>y9h$4D_mirb<pC%YGObc7dK~+GhiOELLTXX*efAQMjk^mv~_g3^z^Kxr4THA8(t!*oZv3K*<T&@*L0tfCb_zN@nR5'
            'r<UE2^I24M+zflcbOGXx76zb`=@Ok2D48r?a+ty)m{l2re8b5DvCsiXZgsE{vvNwQX<4Eqzq%JMji5;wf$}nm!wX<&0DVI)?Lh3p'
            '<e8o+Nf^$^(M9<e^~f<L;yY1Zz7balS-X%p(57(^t#b4-lpY~+Ua&Gs)FG-(fP)EU=spsM^Frr}Lgu<Y-V{?gOfj;^&N=ASWnRFS'
            '`pZ`CmmMA~mX}{O(@xg<)(IzT4~%q3!=0M*#(FXl+2I4MY$PSC10l<z`}P(T&QO{+DD#@72}4J(uWqS-^2JfFdBzsyDo~oqCJEBe'
            'S<0)E^sDg%sSk%aS1*G?y>v05pd-yKPjOesU(uA->RK`;Y}7(EaXa0)ol6c)OwI-LH|JktGFN+jnMTtPF<yRYcR#7+GjL(MZ#C!!'
            'H;RUQh#!~upgrM1dpqr!5O!ri0P2TCbR1H)w3+?2y)lPNBlh+ZL={XDP-Q~Dd*8ge(fy95oAws4V|=_-H6HHH*swEGLwifNO5t$3'
            'K>qedZiw*z0T!(vhnJYj4DaWS!#?Z-i7&v77THFuQ|t|FK@4V}Y^dtN19z_sqV*=3(9w5rFh;-ZauQOY*(#9yKRR}EF<^Ax1D7x@'
            'aus1(h3V_0fJVH&FD~#Dr*wS!gycYWK4*itV_k`<?i0>JGU(HBrXA7mz;TFq1uIB*#437^r!mL<bE-ddOlL!)KpiM-4rO48H`?^b'
            'PU=j``QTT#1>1Q5#>!m8k=01paQw6yGCu;R8D<421^%qdNgU<Kd;phGP;%g|sUxLOW_d|*PXalX#9Fs7<b!@*Cd?r`K7N5LfMh@f'
            '>&QnHDMuD7LU;RRm`g`ctsE49-X(ce1Lrmev;wZF%5%WBJ!Dw;+T@y<hro5w{aTuG_!>h!t->(wQrtnW@`6p$NjgK{)m&U*Y$9?Y'
            '%m-~i#ul#t6;i4Q3vCQ$LHSy9c6<%yfPhz&w)ryRAcS{uiUu={#A>AS0T5V5CUkr6e3>g#j)zzN^oQd<98KzO1LlHo9yp^W{G>w^'
            '`LmJ`2R+)L^T#iq0Aq)Eh0EGB1;(9{<yyMt>xBTyYMAKM<*jPmzR;&c^^Uz4)%!jsMimt1DM!LEdX&$C+3yhZ_7TI!cs)h8o0%vz'
            'Q~OR&SJjz!V&8aQQxLx+Zjv`Vh1-Xgy%d8#*5q_8S!kX;c!3f^9)|k%h%3n*!`4G<-HdMGG*e}KDPFGmCRLG6NS4o-%a%6X-oA8T'
            '?G1pfC_*@ZKrp}+LE9zmHo8F~EF?J9GC6ar3o0hzBEYBYk&ISJ@-z+!3~~3k2ba)to*ItalDxu<V@7Tq8Ch6qR<Lg-gH4#5FVZXO'
            '5XeDMk|<94b7TYgyHBjZZbfJu(wwT21B_(fW+n~BbjPaxg=tkc@?<sxR(YW3doENW?~1Bk?`(7Hf#@ci<V=pey<V-iddq(94p)F%'
            '`rOiTbGPmD*SIg+%s1#jtO|KnoJb=WbU%P^grbf2ayKPHY)e*%nNac=CM3#W<IU21LpLK*uEA9>!vqOUZD0=5TILvO|C1NkHm|r{'
            '*2>n~WJj#$Xf&d+jN~)*T#{V9cF$=oI=&fiWx^M?zXNKL^1$r1XT32J-f*s)PfBxR4&#qzyJ;MwllsPJ@2`~sXYdD7;27PLnQ-XZ'
            '#8s4YiiUX<;^j0296A~Ltc>UyhH8oeV4-$l6V>nejVVd_o9U|&n78Uqny=H9ej1hgs_`<I<mz&X>JGh053iKdBj^|irpL+tJ#)gV'
            'IbA8-lw`S<D;oQ3%vDdH1vT)yvkEuhkv9vL5oMa9{Kw`MO@Iwi11bSqP;Sb6Rl<2UR9T{oE>xuD?=Mf#KSJpLMj`Y+R)Mr)8Z80y'
            '{t-d{j|lpIM9_cc2>M$D&)*z3&u2etb<rB8?##}uxOIzftO_&F__?Foj}uH>xkBGOJLb_KBDV}`uP*;?;+*})f$xfF|9cB${pdK>'
            '|B8UpKN{)P66N(*jPUvc(Oovi{MDkm7^&^m;&<j!)oDr1=;0g227iZXqi)RT{js<(-yFk#agf*@@?HKzp<>0s=MNE6pBgm_=DGGc'
            'V#&?|e)D$}OlD=^{M!VT6$k$aCj0+5m`qOnJ}*aK-=tLqnME9BCc51Cs6d7EBaG#E_Tu#Tg?Rbm__t@`$;tQ6o<Di^+PagMG=P{H'
            'B@lSgLIToMngkob8ch6%;pU$`9PNhj{1t=KTn_^>1=X&~yCp}8%qCNzuDU+3SHI+rg^Qn+9m_UfA^RMN=f`hOUTZ_pKs&K$XAq=I'
            'fNr@2hTcgNvI(P5GRf&4yg=Dj$f%NvI881pBN&QuVLXRpHzTVXC8Cexj&jTj3S}2X>mvD(;T3k2;9VNO9OIQ^xavybN<w7g72X!8'
            'np5l2)AlT7$6}LP&6;&%Afa3k!xd+@oue&!u<|Qu0@Y@OxEUH^)mUn{#@Z{;*3{kt(+p#lok9bh1>1TlHG~zGbB~+wzcC=`|ILU>'
            '(^9gd55`rNPWYusjNR{&24gWdfU6=%kBB$|Z@0#rZL<)z7~4{0uAByq-Ay5&H*}`k<llxj-UO={w+jZr)%q)Aj+yvBd-3?>`N>Zw'
            'r_aQ9CqF)a_S?(XZn~u|$Ig{>1k8Ywzy#W9k_9~6V3ToXC)my)G5ey`S%7rCfQFPNswwjX`T)9UVQbNwrr{1=1~d>au`_8h#aKR&'
            'Jn4+u05w^rc^R`Mq6XZ#?&l8#Q8sk!ZxzDV<SD0o%NstmR<~X1TnP5pkBtJf?}GRn#RFE8UpEYU!#0!g-(NgodsyJ-3I+Up;eQ_&'
            '^5<v{(8bl!CG3o5F_Z~Vb%&<Gam#ho4(Oiky!xoXmp>Bh(jMQ^7SnQ@2$lan{^5VX<@=-j%m2Uh51+>97H0WRX&2fTX+K?S7c=eN'
            'mPnT}zMu*hk-{f70?;%q(Seb!dDbR<mgP`9@x?OS_*$<F0v5XMlA&D;1_ND#wbopNwNDr<!3q~&e5F*^-Yr0JCP{)M%My+|rT;Hc'
            '-Yv-'
        ),
    },
    'video_analysis': {
        'sha256': '719335b9ce67dfea3d9d5c4055d03f42981a35651571f9c11eee8b4de554f553',
        'encoding': 'zlib+b85',
        'data': (
            'c-qZ<YjfK;w%~XD3XHmHEv6+!c6a*Db&q;0lQ=V}$)hVyy6>#lB~ub*aZHiAeArQPUA4b#zijRQ+VcVkfTXNsy0@mQW@3@V0dR0|'
            'aNg+m`|qNv2<J(pZsTQ?jlwicc12vM$7-FGqfNGn(o!v>GFp^zmZ~JXju)y}hk3MA^IdOpJfns)-EaoYxYan`?$Wsm)1{hBX1lB^'
            't9djFt1{b!2y3<pi!#djvvIHYCM%<PmffhyvAT((ZK2|_P}yCoHdRu_qXj^Umth&Hd6X{Jn=rqL(`)tNr5Y@<%{I*OR}tMtd6@Km'
            'UB|^X%GD-YMu{roPf;;c#Ue^0nM{_$0~}kn#leJ$diClxe39i*UO<B-bhD#>gJEx8m8x7vDqNHmd>JO!Sss_`4Gys$O`%33rw@P&'
            'eN@Xh8*QWTrnik3IArb@;U(Du!145o_!>uHVI2p9b6RCNd=et`B)*N*I?5weWW9M-u2m{nD-?iA2x{~Z#;J;M6R8DEU&SflIfR+6'
            ')FzD668?u^m(@0j7l6-UFHV=yHiG|4%ValN=JBe;FBaK0jtW?s_kc}AWd(GSB8C;T&B7%tPp-DrJc)~SRKT=$<r*6Ph-k^uWT)~7'
            'KMG}{L@B*h3{VgQcYs4^#qYLxbQ?!^YLUcSq&}QT9+g#|0-?p}XcKL+e5YXjLsgV{1uKz9<K8ovG%nb{;2}^zo6xk_W@!;4Wdr`G'
            'FPsvfcgE-~#{VdQpvz1x<KkxA>-YP;Ri16sY__Tp_-v-)%{I$R7%V_1nk{-gd6q}|ucDf7^K20ng?>`Tn}}g;!*ZR(a}DTc_{lGj'
            'j`1`7M*kjC<4;*?z^}ssLFk`2Xfuek+H7|!EL6Jf_0HcvegENn_RG^(AI{FzRDJF#_^%&*gnba>jKicqRQ)BaUy^O{!EBp^i)fuC'
            '%P7ZJyacm6`p+tc{oNljgm4S<yT#wC542RRR`DXn=C~O6a23Lm(rfr|SrOsSaOrmd;vPo!;pd;<zI%W6e0Ki!!@Fl^vtM35KYKen'
            '`;Yf$Z_Z!7ePfMwyxD$_fX18bmjAusfA6CChW-d|V}9}dEkl3t_8Cn3-PvCR(x6|IVP4`#n|QfQBK!-88~<5U1+brf(ChV<(F&;~'
            'OKzi@dF5ex9St4<tH`Q+5zUrW&b+FK7DFH;Xb%&+h9}cEuttoaj{lI7YXF;AnkXK_j0Raheh?CW5963lh8}Euc=G)uP@|tz2|F{o'
            'Si$ZrVZiv$<td{Uc|A=R_T#$Rgz0FN!!j;mn<iO$J)+&Icy_ogfZt|mjC&K{XXz3d0RlSv7#Fmc^p+#egPRb!Hg4QFmnNfmSitrT'
            '^XrPYoO%!2`WcW0@LvM<4E9t8dlWh|e5!y!sjDj++gDd=n^eGq;<N&u!Ek7?6mX_6xeIqik5LtFw$LSv-(uLjn(&Y6CvfPf<B5jU'
            '$DUy#&}no7+ZA4(y?rs(7^8pVm9z3NPBq}A?DvCHvv9OPK!1evfxBkNCntgVRIi--wY{F$0GHG>fkAKwe^L{CX+ZpGswSrn_2gk('
            'MCz9?siLzy&+<Xv8_ZA8L;)Q%d;JN4Cz~*d`*qjMhPnHp>9`d&h)F<?vnA`nH3qio&*LOcH_<Li9YCk*v;FfA`Mr#%%y$;X-A!t+'
            '*|K@<+5Be}0ii8t53B-ENT436A131oHI6<mqHU=L?{`}s{?M9AfLs{9*fZHaZ?FQ6mLf*<BiD&P=?6u)iwmGtjebSv1c20t_GjxZ'
            'e6zdiQ<d_OtFmQy9V_74sk#YwX?UZYvG5|qyDS$h$E!9pJ=CE)pJjLx--IxoFo*soM@!h@^J-V!xCCY`0wCO9x?#&Wf>z3#PL=Te'
            '=8iNgn6$N$L!#NtAB-BA(FP!)oa}{)2Tl`YH$(wdM0F#F)>}A?caAv3`V|D*zNpzJke`8#C%|k$^5<23kX(I^^|igE(KTRag;cZ&'
            'KUq9Uqk?FL8Pg{GIDqL7_aHf70K(AntV)-Ix*38AhUz=`AHS-p@H;Ir%xLVhcr}A_V-dl*w;X`ba}6Y4(D{o81J8D`jutl{Do@pQ'
            '1kz)fn-5XcYJdUp1lt<wmgY6)<iVgGN|nw;@|^L@<zTT{4wV^Deewe<2mwNzER{4-cWV^H*ICM11|F9SK8=q+naDB_8)5dqr-011'
            'Ky%@>2mn`Cpl@xel#~ZJnKwX)Y6&xf4UQ_wHr|3*|Lb30zB(hQx<eQ*oZLTWNkUppoEIg*LtOx+MSL2rKqb5j^JPII!4e*WesBYW'
            'g%je%3pm@aRkT6HFiNxPdJV#Rc%5b-kSq!~^a&m)7@(ukFvcLQpq2yr$cOhYMt_8UNaiK|Doj8iK)nM+6V}3TWGFDa{@X7EY6B}r'
            'DhHmp5o%MQQo>xqYnUNX{7uZ$3DvKlMP+vd^-J>s-PVyK5D&-$jJ`1b8IYU~>J`StE6OFP%2WHRmp`A;^C-_7&ueZGeJrQ%^9sZg'
            'P)IYx=CogxtI;2UvLf7O#k4QeZ#ztx91wuJSsPr4w;v8t4+<uQV;WmzF+ep2=P?<d1dZnN9Mxg_iSR0GcJCPgtRLzk%^`rSRyYs$'
            'Zq0@;Q+M<%+knUnC(O_M!0e&Lm_Np4A9_;`Y?0v4AmyIXpD4Q643MB~f#L@N3aT}8Ftfl9faHOXA=ZL!=HH3o?}FafVUZvr{SJct'
            'Jq7Ne?<r5hZb_mT*gs~=I5*swpP)WG1*sW1B`U)W)dcxPoNlX9B$T_jT*LSk>!iHV(14Jlbpv=i8Y6bdS%fz|g-3eFUtKl&25xl+'
            'Y6i2DqT0fj(Q>TbLbv%H9!l7M8r?<0nm_{@MVoE8qmM!52GxQG2F%Wb53>dEu*yZ0M2iyPzIUPJXg`4u&^m-Gj(XHW4K||q<Cyf^'
            '0yYw9%?f$p7N$>TDC-P^;o!8?JX&QrOO&X`HuRSUB2F!u5p+#yMFx#e9UTM(eBELU3Nwy(5VRfdFW}9atbBndsh0NtW(%M5Dyd+q'
            '!1qDg*D&w<PIsm&IGoEt0RPO;mUoEK<M);o9o+~=(65+|v<wmFt$2aw=Ab6+(R_|bfq!hA9Egt_P))PEa`93F#v!v@q3xgr4DF&<'
            '^FyEmG<DG*K%VxPfZk|O4sU|*>LoO9knHFc8G1gBlk5(K4utri9j66|yKbOg(l?hj^6Cj$0<+m}AWNgU=N}nq2l#04;5dTdLF7Bc'
            'v+CQ3CPir(xvEmVDr5B=6_Oj+sIXzHPhkpjLzv@Y=|SK34R~2JAu<#;>*)^FO|+YWXggnqgsxLH@UTX_VV1(D^a5FfUJ;mN?;CZF'
            'j4;|{c=U|!vV5u5QI#`i0SO=Z8OV3nG1`_>%fhaW%Xm<ZduOlTz6Zq3etrAyxn+$j)*yQ!`sok&9RHq2WghP8zu2sZ=ZOB7ZGj-W'
            'm23IK*b+gsBa2~ty(SACsOEqg{Hx3>{+tZV_(wR;=vaiA>E=mxOXj>apif&KLER+NBVvWRL$eV5HBYjPhmwca2n2dr=WqsVaPc}!'
            'x%VQYd1gtq;eSE)r|02qSl}Pm@u!3gj{tiS(Z{H+Xh>zWPU0IH$W2H+MgIUfVNLeYGQ`QEt&$E-5Y+N;QO?NR0K_mO7r|h)Lg{uk'
            'kJD4MaDx&O-dSvT$b>k9(@=g>v-<Zw3!*Z{pHn5aLEAcF9%^}sY?7Wn1INj;3Iyc~18Z1={njzIC_J*Wl{W)mA!VjQIaqVl8$ZDY'
            '|LgeVhwmRhefo^-QF0Cnj?i~-{KL&(KdBAqDeBj!zp&I@0s+wHEbs%@gU{1qk;f#1J!YqiIS$o3<N}2PH3;_;aR3r6pjnlojI->C'
            'Sc~E;$Q8U8O6Y5>k5YFD^wHoxW!~d1j{eep8nnP1Z^Im&XPX;j*!-i=ay?pJXW0$?Z0qI=8uyH08N^-ZAgSpPNOwdVeYA!#&gNm7'
            '^2Qn^*>!?`1R`|AX1^xCZ`4Z^;6~&_BeN=qJU~-0mKc~2E*9FG<$Quu8Afluzq%R~#nqJs>@KT7X=UY$7k@H2olI1<MI-M3!9^cK'
            '6yXCg>Z+6PwLm0p9}65ZL{dG~d{=^g_$%saBhr}Au?V~sMOLk90viQgQY77?_O4j~je&|hH?Sc#Th?M}?xa~~q6SBch?HF5`l}o{'
            'xdGFMS@&@}<pZ;&0z~u*^=@K7-(E-QwxmBtv3%?9uV84wE9DbsU)mpR!Y#s@oZg;J2&j1wo=*OI-~RM=MXV5n(P&B=W)R;!J_#P3'
            'oXW2?{c_qYPRVbR#Pg4^y{Q90K*8J=h4xigl>9jUI9qM#?_G8K{mEqO5YzMQF5Sc*5sT<9j<T_Os-BFGjZweKZ)08@q%&T~5M8(*'
            ')nbh#n-n%Vtuv4ZUOoN$+Yj$+^*%;7WKq_h*%TO%LY^c5LHZ#B$r`O3qmc#BI4-NQB<F69j?^5zwZzHtXjLKa0STKp&e4ao7H(vK'
            '7PurtPr2&<U<Thm!Wjd~N^-`WuhH{|UJtch19Jv0w;av$?5==Lfpp@POP^u7y9;+j)QmrDOLiKz_~byApjL$WbyS+^lk--FLMoO8'
            'a0v?nlof>=K~9bkud5I^a2c`hC0w8)YSVYPzy<bcgpS9M-;KiCEMCso<Og~s)A}a7iDsXoJYyQoZtD$!6eO-b%PyGv?09HNoOwik'
            ')2TV=L`j#w3zA3i$Rks%n!Tx#<Y#$3FcN=TgHsDbsP+kU-jPv=-CHFnhAZGPAO)s7?SLX)I9BIX4(k>P-#M}=b`ynYQeO(6BsQbi'
            'as?coO`OXt;-Mrc-T{~V1e6Cy<mB-W^&$}S94q6%MlzjMz~PGhRtG?r{qY#>j(kWA8Zw7{c<9sR;>qcyC0g(rG{u!NE5pQfYO<l*'
            'VPOqphjXTP+qCV^K=aZnS3Hju2z>=e=T%ZGql<grqV1aI1ug9+Sef=vsLduD+oYua(Be<|l3@}&7FjMv-%a3HyfjK8%LTvn(O#cC'
            '{vN|qB5fQaPE(frkAHxH@^+_Pe@uT1C)h4Fv*P#%a@k`qp15T1<b5k(CjXl)0Xnaci7(=tFpnQx0TT1IL`p$VDB$z3SazqlB+`jV'
            '*hek50D%~>zJ&8+Q9oX6tA3Ykk8w}HL}uk~TQ>kH8%@4E%UPVRvQyyqSpx4~gh0hzz+Bni$o%ikGzwMEfX}iOMR1MoIM@orZ0fTB'
            'sSNn$3@y^Ij?p#w{&@}VD0nFf7i;|jnds*uBlR2&)e$fo{U^e}f5Y4B8BivSael-TI7+h_^cpSJGk&U$Q2i~APSu42uLegSz(4qX'
            'h)&OPcXWSw?^?@f*Ym_?GBM3EC8+E*t{rbP4lkQJZwB?+>cyP$KSO81ruGlUj>gpt4K_oCYll8Agg36qY2ci8^_svgME_Y8UPlsY'
            'GKWK&1~PpD>SMS>TMs~Hjle*`7kH9v&@5R@KlkBif>sHf?v@jMQ}j>OlgZ?Ms1S`aBkdL*I1K!VsY!-g6K0-O1rfuP>CbVsSdD|<'
            '>WJ28MBaC4G(<yhsf*`&|F?M%YzAV(9*0|um>bA%aj2lw2wTwDtD@^MJ_8;Bn&}|W<FLm-=2`DMXBsn<?HW_DL0C>4Tp9O!-G*#K'
            '7`lR;(N009*2Se~yoJ%=%P~7x#b=?U{TL&2TKyQZ@E`$Kk@LcgNF(fXqg~%@R-iEUyz;?c4}{&a&6cvY+uFgau2%RSn$Ty`2cf;?'
            'Q;Su!bz;4y5TEe``uyGlbl;$!JqYPU1}<9{E;{glP1@1WM{0wweZup;vbb&wXzOjutots1ZfWm+&<29YR91>`R=>ZuB30^x&Nu2i'
            'M-W3$9=<ip?9%}{(7t*EH?-bTuotc|{A0E={A>pAnKrXF;%Q3EgpZDH9S}pJuXZ}S@9mwnh$I}AJfj%pVqn`Abx?FIcEDza*6gZF'
            ';-b80i7dRd0}KlF^X?oWKPOl!ybdv<Pu1Owt!|E_gCm^J5oqCCRQitiEY@;FNIjz^abOu8h1dfg1V=m;W9R&)8FFX(@TQ@?4pYD?'
            '9IfbhXVXb*qp=AR*Ug{G2%pP~A%-sE1oQ(I2z0yIB0(Pf+2HZi{~i@zfe9~t@b_b&h=GP?Xx#zdV8r|5>xytqk?G6m1|U=*zG`TH'
            '$nVu)E*_0jB|{!6J-$F|6|9a-E$j_yWq?EwIk^y%vkfB~zPE8dX!&FuE|)Vsnt?sZK^w$z3mw~^5z$P&Hh|Ar{dy4S^;)T!#J7@1'
            '9o2$hPYjl9G2XSh0RtB(O%=5zHlP7YGXM>eeIp1^_Py&!x)_-L^r2uLAj=m~gCnEp)J`qrTnazX5~#?KQFv20T1uI1XFw3>cb=-_'
            '$q<bevn2pP@yK!<;M?(&JsyZP00Qc|Mi&=Ddvbh2tYsN5O2b87QBHxSWq|lqkZqT&O5_O$BwmO(1U3A8M6xId{^@p{F2g)Wd)HCE'
            'X*{ACvVuT<HP#C7Ly&66<MHLaIDg4FH{U7t%)_41xe`mXyrh{b?=nTuDBA#)AwaN?9mA7^hkzQNXGO>hi6%MR8}e|B`4oeOKwsoj'
            '(wIdy-V|1gRTNSZ1Uwz=rU0+T^0fXlxR|gYNJz4qqr?w_c}AlA01y>&_&c;40L*~9FY+$HJP*VI?%gm>JSjC^enhe(z}JXVf_ia!'
            'Iy%0zOexr&Y+6%mR}~k~`qJ@NgLXU!=f$9IFj64DZ!Y?JE&G=N#y0!Qz=MG=aWwXDJ<->GWrVXDxIV#6{~RjV&q*_~g&TOn!yl?2'
            'f_rBYh+Ag}7%a-$tk)hP<&283XgQ5>R;NuY*qnlAfJ3Bz>j7`@Z=zxNW%r`}lo(Ws4(nSyP&Bgm2dot?Ye5@JK7f5fYAxY_hd|zx'
            'b$q=xYs;UGF>Dj*%Y6ua1?H%uW+Mcl{-lnh(eX(`x`QF0p@9fi28q1#i8FtB!=Ny3cThtPj7N5heppTnn$2-3!#s|PLDOw2!iWC7'
            '*XgRV+bB<fl0em+8D)*7Ddc;X4>V_nT@Oc^vg|c9(LP2;KWtEdW;qB!OpMaP^#Q|5{V<+9G%yj{;n?P>S*<7I@18u+d<9#19xiUW'
            '0Q}pNj#QQ*9T&CKc}D97Vj|`3Ew!~>2aQpt8i?wf4QI2Zq@R<l7nc$+c1}}ahq5de`}qVG2w8EuNGdo3*l);DU928&!W}z#Hqey7'
            'k>dpChB2(<2~lYj=9o%IiGi}1z^pmml01qq1%!k^2!Q6G2xF{%#STico00WH1FS*8{Lyx3@Tvf>_-$20OBSdpfzO1xl7}Z*5}>Fk'
            'M;elvN9z#HH8~?mTw44+utV)h1$MefvV!t0*IB8({b8yy6G(TRWlPMqQ<4no*x2l#e;t0!te`>D!Vq)}%gh9R>vqUWM5nYor(Mkv'
            '*WB2JXv5L^g#)E$(LJ?rqp!Sa$Y#H=w0St2M*3}I`&crgXR2yrAx?-rKO=R)>MgJ>WtQ1ox`ruw<A>ST;qKlfv~>Zd@a^wC1>INI'
            '1Y{S_>;!-de=Q$53nk~FZdsGSg-sT9D@4wvta#0QDSLy$&py*a*VuYAXp%uqO?q=V<U}Yk+s?@5b#H7C71_ZUQJv9TnD#y}>oY*F'
            'dGn{Ki#;u=C2$hr+Wd3Sf127DzHA-%P7vQRp@7Z2&54qgSEsc!2D=jyXq!dANkT2tBbX-h#$;SLq%~?8{hzcL({YAc0%e%Q*J-pg'
            'blL#c^Fp&q7)^AI^XZ#!9je&%TO5&oH--TH$?&+_PTnBcGZeo;zgK5pHI`!J?6I#v7H`ca<XWLd?qCfsr70eaCE5B8ud|2z?O|y0'
            'N6~?#Z`AW>h4a0|_>o&6y^!QT8t-FfHas-qlqdp__`uRAow_X72sD;R#_DCsDL0&nPf|l^L=q7j@Ql{7g%(Hn3~o>iV8eMf^bXsg'
            '*MdSIGwaFFIbZ`fb!KW=T>IM$^k}q(_IdszLH}qO=Ug!%Oh$yD(e1JAK6c0CC}ckOw~gI>dtmLYX#7qaIR~6A<g|6(>~HLMYd!ZH'
            'I-8Ho(e?xDOPl`SJNqYpb>B>L^Om?ya&wjuAM1q^-H1qVX4Op?)S;H`N{v=DNyFT)!*uBhovMwO3vV7fma1<3gSU{YZ+s`&e>nz1'
            'xHhfH9!=<<Qlq|FfqGbV+B;wo#b$(5yWPp2c5yl*#bE}sMXz|f0r0yg<4Nr-aP_oj7>jvWu@h5UMGa!CM?x`g96`x$e*!ZiFbrjM'
            'dvb(Cy9R~Bj&Qf?vS?R?UF${c#Z>Pl7^`R5W*(;?8wg)s;*yp-A>a|HLiHoE(80Ega=qc40DSXZy;1O#S>b#z2mqOT5`CnZE#Yv;'
            '@oPIc`Yyuocu+VeqvOeh2dOX)p-5s1AZ0R^Q0ePDik2~)8=QAkaAYi-{BT!DAJ8#oGg-p8o||w0fvU?qG<`63L7*5r8o>N+hVF@i'
            'MDbXd-q22^X};x&VWdh};7MGgivuM<@K5n{hmT-|2-b6qb9?jly}Cy1V=W-JZVVRSi6ZlMCT}U!z8D0q7D|*flUdiMOU)y+l2;o>'
            'S+d8Y&u32{`qZpN{B+;qKa*pdwHV_TTCALY$PQ?QBDD?zZEY6-R4XUO#7|aE5Y3%dr>?vbEwrfYD$6j=dgwumRjHPH+7vlQixd~P'
            '9iz~4SorY<@Z8cd)lb%(aI4U3OOIB6(i1X(E}bk`F9PF&o!yuAs9I;Dhr*v5FRv3^QBW}Ccm)_kdvxBvcslwY;po%<`Cp^|`@fGa'
            'zwHlc2qs6zjm$uAZWCkp>cWXMxOB6V_K&;qt&a3SA3N#P>_#7Aut#gG2YqDs-&DR`EVGMZ6BajECV?Fbp#Rkd81>)uW3yO+PuuLw'
            'KwEhY?y^xLMH4RUI!(qSWjTY7$5xRPXGec&e1&B_IxxPXHWW4ED-7AA;T`?)?)$cdgfH73x5Pw%bXk#4nfM5<uHU~17;+RjU#iCr'
            'a{Y!R($wvWYGd(=KcP*qUQ;4Be9~gUc%UC%b3Q}+B1NDNYK!Srm{AuEYC0LJf1d<_$2ifMWUK@JdhvPe%|mYtH`~qLG6q9PZx(XZ'
            'arXkx>a*RR$R-N8Iq!NNzQqMPMlOQE%m@YmlSWcRrHrP1d|3Vi*}JpQCv;@t9&sXPdOCQNkJy`N=W-=?b%s7w3>)-iD<>Z6hh4ZD'
            'woTR!@wIZqK!=1uo*I|b#oYz`W|n@33+L?oOSZ4U7x>DZ=e;Fi6|QdTktE3BlVNj1kgVLY!6xJ5CwtH!A9NEAyq*l(*B=(~iLKa?'
            'KQmoMABV<Ek;0a&HYD^699&pd7GI_^Shaatjda5B1@&`@#08@0P@O!8+M2^(^!*;;5S})M(P%}F=~EsUZCze4UiC)}7&SG>$mII~'
            'M_fmGIO-4Vlod#tIeutiDMlTqkaCxvp8;D~qKY(j4CEu(nk(1%Z$7mh0+!(olX{{;+he?b&{^3D(Eh5W58r3%Opp19?tF-D)_=Fq'
            'C5)EUz5#lfgN6@wz|Ln|KKoW2(iT>@T2hZgL5wBA4Hu~Eds6Z`nc5vvcG9J7&$kc^Y7(UBaEha~q=jrJj4?p5t=HnD6{0Ys038HV'
            '#nJ$dyV{QFeC$e7<cXXY0~c!sUQp`@`W3@(mPh%?SWfq;-a$j~V}OW03&~a{x+VA3C~@NsJTsAF9MTq6ZQBB5b~?_R!ND329u-^9'
            '>wVSTdc4iHgLdhO<lCmfsR^v1;S9TsIn$2kO_)uC&b0Zc*{sc+Gi`h5Hg%WYLLn;T#5qw4u!sP)(c`X04gKBXz2cR%zaVk<Ay_q6'
            'YKNr5lDpveMgt3wSIqER)i7;70JJ%_PLN_0?E>iU2m<LpbQ*u7p4vHqb2N)pOD=~CLR*444x<s;_hV3Q=3!YTXfaY;!C<VOqKx7}'
            'WQhgf<>cH#xdqKN#(%)MpA|YAbl*rPg~Eyjm>=rFBPYJaj;TsASM$A%!lcldjGW~h>V#J~lc9GcTn`?l3Z*!Nc^Q`&tE<jS;`3M~'
            'nA1>euFDxh_>hW}da$^lCyWJn9Wc9M+_g08u#o6-S(GfPd2y{4izXFx?<g`(5sa4DZI&6QpX&xkzG!lEd2wV*mPeQOAm+7zB0lpV'
            'C`TOahD9@f7?>l(c0v<b_Cau-%nR_E(7siJexJOp9cx8;QD6O+`rF$#AI@JtReyi`HwAp7+=ZWFRe{(6y5T1J6!e2VJkhT6EOvnU'
            'B=!UH97g$2Jf6rs_6f_5k`AL|qpXJ=c5gv(T__p`meAT4Oywl)vFByBilby{0{MlN^*gG0gp}*1{_SM4b!@3*Z?5Ba*gS2I3V0OO'
            '#FmgC%ILF<!xU<XrRF5*UAM$(AE^Px{Od=QWO@{s;)3-YuC9zt`)(a$loq^7;uOn7I5oZELLqa+z${pauu4j)TrG2#dh%Q>EYkIh'
            'xww|9Qp~i!jsRuZY6WU`Ar-{KCFT=PxQH>wGh_v2wqZ;dm8(EDMFD15<ye=_NA!)=iz-P*k_B#Ft+7@&^KT5FN+P!vacba-Sldp@'
            'P2}O7RSHeX8P?aC`s|yX7g>UZFHA+UM|h3_V(bZFG?=cj+$a!MBVE_<9T@BlO|QYBxdt{Jey)Fbi3ZYvt`VnqG#|tu>vOavYIIX9'
            '(Nx2^w*-Ts|4JY4zg0G&MfAFii}k4S&7qZBDBZn9QnQ9Vro(A48S=+NAl0~>_HTev`a@lZa=PkI9)G3}?qL_ipO8BWCQ&I^5@RsC'
            '*s>E?y-FpY>2YA^xYt`&2e1el9s|zn*0az2@5K>{6-Sq+lau8=6>_iRqWFWp4U^7P$G~j5u)+hmq0j;9xbnmh$)#OBvte8RXEWZt'
            ';w+8MgJB~n-Ydo8mSgd!=zaK&dL3>}DY|4gR=<kygn9KtU6uwfl3@10u1G1zfgHjEhu^$@d;Z>36CoB8>hcsQGwzTVQbCYVR|w&t'
            '+9%^<A6kylrCgEcI+nN-eQGS@)hf!Vuqq9zgdS`51|U6PA~phB8mpuKek44a@{V;uy#j?|tf4dZ(0Oq2nq~o;(a9+sc2ad<PmLNl'
            'MDmb})OZ6wXC$p8#D*@~U@=iwTyvQEDF((?=}n4Pnfx&FOTIW|U$EK<gQ#oI<Wz-fC`TY8pRt5ua8~#gTHLZUa!!D%L3MPsx1Ekv'
            'pg$k6%`z>mrCPe`usAgCZG_9is%Q9B8#*dCOdAy-J9$iFOtoV)@$DZz5atJ$y27ol^>9qJ9(uM`2jX9aq6zVA)W23(;B$>deza?v'
            'E9*oe7{%7^V5KK?%1Ys-DC`UYF3%^3rS3Q<1oQa)hs0<6=)Ucw==Huib;OkdRnxM<cyBB{q^~_-!bI>Spr{k8#6_V+7m#oe90-e~'
            '+X8EmLd|%iL`Ij-pc=ZyOCN+7>1GEk>C7SWqBe7!1lcALpoW5R`leg12MZf&xvCP;c{g~J?F!g5Xb_7+lw*+<(WvZ(93EH3ITi#<'
            'Hsjul53gR$&YwMfBR3b!o_~1vlpICpXU{Mk=#LnN3SiC=Kbwq%#A^}<iPy0rOETqMNF}AUWY2f~R7<q*3O04c9)!pDR~Yo**K_<-'
            '@ROyuCK4`)I3YA~@C%D}#9LY>@cP56_b-2b_44c;4FAM>|N7~FSU08|JFnipK6~^2eD?F%JNX2IdvKV)l1*Kk7Os<Q&YCz~zC?FO'
            'i6ug!d#K1R$QsuyY@4d3xP}Ptt}X?_=^M3B4KfCDOUBh$okt+s&Ja{2HTJdOa%Io@FR$MI`RS|K*_*R>fBF0D*Rz*@`RjX%Zu;J<'
            'yh!Cq4y?N9RatzfMB*8DK`8q5uWw(S$x(JBd5w5PSQQO_e)gLD`p6YvPl*!rGj6YA%BfH*e`BUmv@EC`S`_(2%$tTg;{94@rO2w#'
            'o^UoL<$#hI92CAPFR)+2Kx^GZiIK5uC+fP1cao{1r3=<}lw2@oiz=@Ead@r7uP%4kCLjKG)sXx;eE;98A^j$5&KDfYUsp;=3hEYk'
            'J;;|-QbMwy*)}vBAeAk)la(o-)lW*EG?R><Wg&Wk-n~B8{0fhVE7SZ+g^N5ZP-N9x>({5h4A~$Miy)<U>hlPsM^dL?hb?t<8pyOd'
            '&GuAbVlN+5<y|(y9VQHbrRflD7rae**c9hQf?c83Rl3Y^yW`z$(RGM5E9Kt0t?&rk+RPF{+3V4WbEZcyuMsDO!$j!TuLx^2@o-ok'
            '#5^+kgQah|WesFIsaOtuid8z(Rm2si|9U0_&Tzg9Z=*UxfMe-3UV{fOkTIWjUl&6`MNSNH?L?&2j5}Ww&%yzY>DGlts&zk~_vuy?'
            'W>NOV8>~99D2d$bIx18B)D94fm&Cs6uW%vgsZ&G0jGT{X+PPb3A7e!b8f>eR2P>UxlL`>lXbwUcoGL@9Se>fR4Cs;i<2}A~3#`*%'
            '4!J_=I$B^n!Y7sGDfR|B&pQ5OUlT=iG~=vtahxAx4XbiMojhvLgJpEbi$|&r1$s;#ypaM;Vg`S>E4j~sJ%C1_8!|v>czVEH6uf&R'
            '>Jk$n=M`$U$mMm8GrWL_9sdxFLHB_N81O^W`|bpt-QS)3+Yf)VZsIXlx(Fftq>h^dkz*3?9pV9usbu=&$EF=dVKhpbHb*UgU7Rw+'
            'OTbaF203^zIv&=rI6x_cV{&@jz?L(DrmEGBH`<%p+HSHPcwHaM>LV$T<In9<Z(ZHmL{Sz0nb*-&^t*y%USlW-0^f6^JN$O)Ov1^C'
            'ua9A!8gC`DOY(x9%Wj?9oI7uQu1DV4!qB@34y$*7^_d+i=Y<(^J%rIQMpjsD0g2jj>qnE-8T^1YIWq@xW}eb%I8+a%w$~K^Ujq<8'
            'ES;KiVv?|k<{fQ$mZO2<q=&jL5(|W4`Y^>X>g*7fS8!15kmJ<|hVD~Dd0^YA3%e{Cg0X(0HI=^c1FB{1jlHoEyy{1RIx(znyr*;Z'
            '19=t^+Bi&igCOYc(y8~NX&U2SDe^Ti(hk{XW=3HWji>Mzu>F?G!)gGC64lZWt3>Y8gAW5tW(l2+^L<xZ$j>nd9(>U&tI=XsxWl5`'
            'wQXT&bZ1JyZ{#tEDE0!hQT9KhTBAW0Taw!_iUtHFk?gn#c(O4Xi^;8LmRXAvJ57tgmDw_hRc5KWJV{03xnSRu$-f6!@6Py&3Z<`h'
            '<BMuo`3E86%9!A0oD+97l!QKG+uLBU&kfy-h;DY)-(0cl3vNKvb;G{)I>9pB=o>LgEEb8^v(b%$`>ztT3Br>KtT%!4v3Dn+oE%M('
            '4H`SZDhlU9dT4g@EV+*KBwD4<f0B2hk6S3pa2;DOqoPKe<}-_-{iG(nwp6*L6+K;UxpWVT7<Dk8u&R0t)<r8xMuB*UXnRG5_EPz@'
            '0y2q*U@A*@Y)3S1!?fkX!^hlV&*c{_<ti!e6xA(Gy`fe?`*nyobpHGib-VW}K!4mpugSjl)vBb9`lm9!Z`JYekb0JFmkc)BwaNh2'
            '!qo9<&3WwwOkx0wF71~M_kC`J8M;*(<JIFc&DQ#zWi554DPvD+Wl+|WY<Yh>Qaq@YywOO)@~`e>T|nO_=}q6sbh0mng!Op{b?$Xj'
            '&|FX{g`^%tJ<8;h@src0IEx#-O#mBOW!V~nx3p|6EbqoiTP3P&dK1`I#tOx-*(AVy+&5<a)3h70O^II+Je7rZ7>$h3PiLd`b%)!T'
            'VQWp`pk?TQVoke!EofKzOyT}gQY0v#){e}1IAr;uU}ZC+?s#QWhiHn?d4t}=akQ`}&?g)%?Rapq63J8d+qR(FMl>-ur7(X&Lwg{x'
            '9_wjuZz*5~@o*@Xj=7#rT~V@y5Umy3TNkYn4X;Zm8kk;&*aIxSGlQ(zT}xuv9zD;Cnm6U`9)M_(coSMpZB&fTpGgbXcT<170<&KB'
            'i7u*F$CSwWiD}wsp-sol!2Rh*zd4#to!sB0NxTcw8{fs$^e?@9bN=%AS<}BHk!Ouc2Uo~?MsR-BsYvXoG{KK`YB~9KQIl}aGaZJD'
            'Ap$1AD^&k}!UeWM^`BK78J{4BUFetEd4r0|QPK1t5~s!Wt-HV~l-HEYz76%Iz8l4*HKeSV7eBBHZ_yx4u?3`k2>3PaR4%)TH<n<S'
            '0i}+l=<}0GfO)Jhxug|lMT+iS0v!q~OcC<UvcMz+I%l-A6Un%!v`dp0@8x-IT(Gn@b{v!LHX6xV*oFzjTu)yQtB&y_E*H9><XsAL'
            'q5DM(9eS)IA~FfLOX<!g@*x|SQ;AAp9g8EDI!{5Y6<L{|RA5tKBEmeA89Pq3Cxbq{5}!3m{jiVL;<E<S<6z<xb3h_5YEN69+#{(D'
            '^zt@XuglifelF@jh-M&u<6WKS!~32iU0_sHd67oNe-WpUK?A>+hqb7qOO%nu4q6gQjs%R-sz3_hA7u98PMqRUSSoUAeb@4tho+&C'
            'w^H>f-g-P~Xhs{H_F82$ZFQ0%gOceLs<cB&TGqW80@u>OPluD#AiW0smJ{8&<67)q5w$W0QxsEXKv{mC;rgNRyBX2-OLs^IOspZn'
            '4M0DDA?|nZ2gbADqogJF{DD}3N9v(z=stu}zF=*n4&gG&$;~0sqEB%hGK`BcPjD1{OKu%LP+FCde7VrsOw6fePoE()^5eV@JEhXS'
            'fL!5J&-<SePtBFh4+^OIOA=AhHbTNl&?%teV7W|6TL`X())s$jNKNN-;UBgDYi2I@apSdI7}z4n)4^pAhq&bIrsuH!M-SD0{aTx|'
            'T&6Q?x%w!<+vi5N-;H2i^Q?a$ptg&W%*#DjBOOw7xAQ)PgG){Q&W3i(OcZ+ykLADkG@5o;c)Y9;6U!}$Bh9ELCuZUW$g@^9NMVIY'
            'oV|Jf^4*zzrRR|9zF>iLN@LLCHPR<jou<qyy1A38E@cw<7hhLdRp>Oe!O8cN0%H^;tAh-_+M;g7Q;<9u(@oM+LJvh*n9~+7d(>mr'
            'Qngw|cjo#%Eq_Yjo-OxoGsG)^4YBC2A3A7BqLNf0vh|z+l0q=2I*#zwM`C{>>x3)d3QLqBPxZ}FAkWiX$9Tz>2_8nFTZ@epyo@P('
            'ShN>rNW!xwD_zSt(_$}1%oK9LBRWA#?eB>7y^j<HCIa_BWa!W&7fNj_S{4@5sQeVgk_^T5jyzEF^ulrD+`-v~(d1YpSQH|>mvGE+'
            'Z70ZYxWU<fygYyZ^37im`bgG%8*e$s2Sj&DM3SIAwE7SzjWLerz``B%RaBDxEB9n@YOh^zOjj8L*g{1m-3O{ij>H4{Q>{pboQueD'
            'jCcnlD-E9C)A$B(;6tNJ5Sxy$xN{n>P<I0bPHs|jXcDjJWFJ)OXnl5#=#N{eB#9C6qApOTDPL4@W$}Y6uY#9R5ACG5e_*&e?l_88'
            'q?9f6O0{fK#ySP*!uGl{+0%eaWUYPc%;{fAG?7e{YL$^Hj$t(mj6Cp^nBO39%{skw=%Dqe@y(&pg8!nfwaA1*nTv;Kmf^$ku(4?3'
            'xYp*?67P9s(Z!+}oC0c2W|Xb5!t7Z&8MgQRNbLzg4s;FeZ(#F%?SI1F>=da2qvHocgbTiU{e5NdNwJ2A)YS4xK6KAp9UiO$Qf(T0'
            '?2!EUF1+pQ07|k0Osy*ytUwVePSA`W!N>k#?Zk{6qk-Bt8v?c6>tlh!HQ_AT*t(>1Z_VDY=cYL5O}}w<YhWjG$dtQ7G(BkCST4pH'
            '*a4zLYflW-+jIIOXk6)vxfbSfUAD)d>gt3TWBdTHpeVnk^VA%vKjLZrkE*cmM@FuMVH}aVu43V)Xlnzh1TT)4?SVcp6I+161i(8%'
            '#=y)BJF|%st_;5~kV_X?oweaD7iKPE^cYMq7|&)YuKMg=-$3i<dSG$O!KzQ0q2WAClUUe=+J##QrNt^8$#sOH3{XBW_V79ebc74o'
            '3xm&g_xC#8aut(J&%eN!;?6nL6XRcz7XwcZ%f<TC2}^zbXExZu=+OFTT@USMvn&lZB_W1#&n#msE#A2M<|U7i#uXqSy%}uc0#j*S'
            '=55iK5R!M*3-FO%&vig|#@+bj-Wqo^^_~rTdsKjF%d?^5@R&OMx;-hB?&;w%20HDBR=-Wfu@ZsocK3M6*G@_JL$rvKr3rTU=Q;I>'
            '6V*ZIv&Ttg9DVP@gG2hWWS`Z;s<jo~hOiAdz>=;Z^*_*8?zvpF@!GzT&;C@#&TCXOhvVhr>YOUu7$CT4vjrS-GpTzZ%6K&A>(Y8x'
            '{Xf&v>+CR>He&26qNB5=FoXaYe6FK;F06}pFELEyhe>ch1}=0H--LF>2CBHC57(MLFtJ#Ee%27l*VLL->>kly0L7fytm~9?jQ}o|'
            'Oe5&#Li#MF31ID`htE?p*P%7y2Q+k>b@+9^<e?)HT<$rgbzRi@E@r#d?o4hOZd_O0teez&fAzF!*D==~yOx@kvQGAO$ZeJz=NfBa'
            '1AFI*A)?lOrp@4q4witR8{fvORtC{|0d6-(@zBk|Vdiep;7~{&^Vtj4xjNMC;Sg?OOksF5y7w(w^sLQfQ`c(N0>-;t1mjJR8_wL@'
            '35yH79PW~oDDIrpz6;NiOoS+OyZ;Wupc$dA!}n?^7b34;4xNYOHt+4N9M%${=x@y6pq;L~P;XQ0RAT?M2dmAYLc&X<N`Es!px*p@'
            '>hrf0fNoBpl`oODIo+2?+N@Yt+r%4gO5Y4_hQ*zeMf`Jn(2ow_(8J{49k&g*;T$a!Yj5SpakMk#M%#PO4<X{qn99p72Tl<X=YSel'
            'TNFw$+KX1qt^mwi1=Y)1c(vbasQW0<sY|*J^FrD>*OCwULTu0;B$DvHA#Q(-O2@+H+Z2<_jV>n=65n)^VSq)R-Kp@7uUxY;uS>Ft'
            'Y0J_<SevxsYBFp^!`|EQEQIXW=I;GSQ(LEF`P&y(i~tsGBFd7aIb9q{mq(N$X||KE#+lOLthG^1pH@+%^_1MTA8!KeLFMZZCCpLA'
            'JZ(wjXzh=qi3j_NAAFHA<S9h0vM(=*EI_RkNrYpqc<f7g^-4xJ&3wRg)jD3x<J`Vo0adbC^7l}w_hfGrx!cA{X0Bx%{3iMv*}Gc|'
            'ZDK)3Guu-bW)h0`1*@MWNVlqp^Dy}$dz)Jlwmn?TAbeP>;cZq(b4{{6Xz_OyTrA43xKtOzP&Y|D*G1;OU;+=8erqZPcJZa&VXiu~'
            '(GEQQ*{Z^1CK()7F^(r-*NmH*I$u4>krF%B6o@fSmbQE`(=Tfxcj`&md{i$E>@}<vgT5Y?^+%tZ%EP_2Ve5-2rv5EGQ@Wf-*Lj{k'
            'KYQ`?!>jia$Z>92UmUzo8`T}ORAD&d)b*dbr9r2Bw;Dck?Kxbi+A{$;v6{ABiO6>9g`FB|CUK%k*<S1>tmCp;(APde`(K*aPOFy0'
            '2%V?MdEI-qxNM#;0Xo5|duPd46qsd$M&C7S+}Ep!_U-0-?Xzr|8V3gNL(V<_88{S-fv|W%5J`=lXz-f2p02v7c6NE6+p!nqiEt&<'
            'ev8rYFr4<;Jv+l;LJR*s3-WB%k-H$(&yIb2k0qtpHel}~b~|Wk2(3DFT{W{?qa+1#Rbw;xsp+qqG}c8(y<z_Juh4GI_HwY|SV)Zs'
            'F25V!6*KWa3Q-wZq4^=Y6)e({%Oj4RYs6hKg(_;p8^(q&SOzVf)Z-Np_Hvu9XLAat<r7f`ZVo~RDu=Dk+P6A;WJVN}>s_m-8=@SF'
            'gBtj^fp}`=o=9j!o0PKQsgWaG#Q1&-T=uTBG-XN%NnnXi#8NL%tmSAmu3ABX#a67XFB`eB#jRzE+>l|46sO8YKB&+8!v*hubmEDr'
            '5EBZ!FetkmDrh@>pVQaDgA_v15?aa8##j?B+!Jry3b-r$xUC!7vOp^18j?*{CbFW7Ky;z(M|@oq0MsQ%sbH+NP87>Q+L}b;Tfx`f'
            'GzM7N$kX?OAGtAsseMs8tOlfq2rCguO=@X7ZqS`o9+gIG0y&&vyHWzj<s(ty0HkbT-#;>!rsN)0G;E;r-a9SY?2vR~w7uwVYix*K'
            'Q~mtmmX4jU1B(5Ptp&A>)e2xV-2w@f6|SVz(*71s(jV2YPv5<H`Q|UD_C0wRC=zY9<xUN-Vz|DGuVjwbx5FPb_+GEKNYLBl1c#W='
            'wKv%)+p26%-r(E>=a`!UG4M;_G8Yt81FB=0Rf~1Bq=E(JE)3Qru<khp>DHC?`c`FSH6D*m3)W7#g_dZRrb^!<t)%u307X^;h9uuo'
            'Tw=%9Ey@bE^}d$eoMIiGJo#Zpu6i;C;8y^f@wbexS^i&##Ts;@F@HV?>WVs4OoxT(&I&qwIYpJ;;L#c5-4`6=d}>#585~bezNZNE'
            'lkefIpZEKoVS)#vO*#b6N$8UI>*yo@HP96mY_&z=IAz|pY&pYAPdVktMFP;lIc0zl78Iy+QqPGB)SIf;(IUqCAuvFI8Lou!0X0@h'
            'Ij++-uv`{~fMW>Ocv)?>MXL@)<3aN{#{bO1ViCtv@>~unc$}`>VV8&%#*9Z?ndQZF&>y0Xeu^pS5<NM^;9nMw$2^KbzbaRwKlU9o'
            'dP^<i4O;NZN};HqCYNl`YnzDVayg;ux+w}FBySF79LPrD^?b(x%kuOejYfr~tcjjX=ah-s?GH_;RGb_2{FI7zh}XyKGc38W7f#z!'
            'Mhs|H0!Wz<MjD2?@=-$3c2{tQl<_KDloFqZQ5O0Uo?&E!r-Z9agclUzEW=PyYvlTZ=;$Lfc93e0A*vOI^*gsoNY}P!2_}nC{mmrA'
            'V-lHzm9)i?FdSRE1&)6kaphV#jZ?~UL1P%qQd>KgLi%1w3gDfakPD}xIp-Ec1QLdAir|kLyRVL*vw`d_I^KwBEFy}Z)Ag!AUvhxm'
            '8)B!$%B4Q6MRPb+;62XukGg=AG};%;$8L|=yR%<jp8b0EeD>$R1L_NvVSLV~AL|0Vi+B@b<S?N41~x5d#IZIvIo4ORS&-l*aIb9x'
            'xq(&hciW~3zG-=H4*&e@)!DQ6Fm64Mr*HnIw%Ml`=v1Md60Mj8+9!2o431$1uszEq1$fj0rtknanISmUcyOISK0RF6%iF)aH{{{q'
            'L<-h>J@1*MC+4?JdHUJu<KD8y{@BnHHqk9!f`}g`VU85X$vxhP0xP5|aClms55%z)__4_xA;Wzu_d`m(xD3RzAa#~z7SIc7Zv!$f'
            'srFkZ@-8)|eV~V4000915^p8m-~njx;zPTy3p=?>YK?Rsu~NWNWRBo)m&V8$L#KP*d3`TU%G8!ObG~qfaM^3Afp5};<i=F1(6oCG'
            '0H2+9e2{qqs`HFkv;CF~Dpps>-kqBUNHiORHbjD2OO<{HZc9@X!3X%y%#=g(I<_DjP!MqCBj1#P&x#!~U|@Pw*Rvjk?X7Ac$`wQ#'
            ')w-jEiu$<6_`y5e8!E+Cqs2dskYw}v$TXYW8~Pvedw<*aUxc63F^PHcr#Out8}$#p2OD^zU?br5j;Lbbl_R;m{W}WM$yr5rKZ2@s'
            'Y8M~qYhi&xW6;Rs%Ne$1hjlKp3-V#UY`)bR*{RaREGH7WlIV`(wDy3h+sHn>Fi_A2*m5MH<;>nw-Jdtjn)}<^%Dhwk#Mz*Y4}x|G'
            'b*Yio_WHHox_NJ!51{!)7mThU`0pDtpp4MQ!0D;~tyygMV|l#y24lB{^{U}(uzBKXv-Ez0(~+6DnX=Y*83cfGA9Q_fc$nS7JnXVY'
            'x16nHpylVzr$$Vq4_F**`yedq5AYYB)hGs|MQsd+AO-e(f_aata)F#Z!&*zoqZ`nS?X4D6`H?57>avR#fNsqfz<q&Q9CW}GHWC8T'
            'Xj~ptY$_DV+{lI_UKCag2OZ7CABj`@eD90`Rj#_%&+YV}7XZF+HD#~-3Ejk-Z@NAVt1^SNi5C{DJ>@Xs21)7Kn?~<x*N@FlTQT0J'
            'H1F48>%u5N3|3tfNde<Mj<<bn@nk8eXuBQ(>YftFL9xdPSMUVIFI+{n-9jsL7nin56c?ENh=!(t0XSqMN~03yMdQFyX;h?`qElp_'
            'k_z+2uH&SCX-|0R+`Q)>kVF4e9Ga7F*l{TBIfaJq4j;Pv`LKO8^k~y|N1`Oomu!)M%ky(;OE_0>)imNpHqlC$Qn^i}qI|QA)X`_Y'
            '5$i}rH${9?%i0aKnwN(ky8<~HFXTvZ@E%}cYO9E`JErE4+~$~^Q&P=pyHsSeQ+H{p&!)`NIS8k<Y7B{Kt*dLzu&7TXK|L^GYfo`|'
            'ZAe-Az2OQYJNJ6EsW0%@duQLa?E<83U2xJOLZiBvjvNvd_NvEbzjHMm!%mH>*R9U&4a{#wU;CJ7_SVLh59_d{<$-;+ad^`wG5T&Y'
            'Yc}N;rq|!yE+!+AAX4X}V9zecq+(`&Vpg1yfqVKzl0nD**~1}ePqp{miR6Zv4{-}Rm2XhfDwk+2R=Vu(Z<spJY*(|Gw$mv%#E#e9'
            'jZ;jhb$7V^89Qm0CFUXH+97?2eax;OAxG3MyXgmqoI{pjsL3<i#P>jqV|zx*HUR>w?L|?FnUg3-MHrkJ`BtM@QEfIMr3>v}OrNLv'
            '0A%R2;<z20SsserF^#<y<rp9k6-iXs2W*Em%N<jTRqN}mQ_%83TU3j+sbi^HOiX>d=v614J!O(G-feLDs<AM`fdePPN}y>TGKT`W'
            'x_^iT>(Ch+1C{HulPzPr<yvDkJMSBA3)hrtL3K3WVRe2??O*3|0SVO9B^@7*!+f|GH#F1q)QHYJc-V~)O&+c53v7A<|BUz+N8sKH'
            '8+`A1MRhIs&Nl*aWW*#^2#)x!z?i?QM>P`18;h6GRtm;TmBKO`X<_p*$A*?1UKTEQMv3b@)m&e_(NWq=Si`2(YS?0o8#c^q-p1m('
            'GTIDJ8<qNPy}HFxGpXHB4bWdKS5>rRFtoc0_2iL9PCr{c^VPMO@`|?MT602AR}m&G%AKmu)cGIsd-oO{9HEgUeB@lJpOpX3F6sK9'
            'q5bf23wT{IM7!;5>9=jfezi4gz|BH!zvX!KwRY{pwJ(;+3iMshGTt#Y_uy+WF+d%!;qdNZqk{_CjHzA#f9ikJbMpF#cepT{oT(jd'
            '%Xf_^ddtu+@yMsDpJa<L8Bv)cj^!QQp0peCJwCD5o=;HKI;wIMlNVO;KZ}cCxuYLBbI4L%1%9Dky?TvO42QXB7YWA-Y^ymOA8UIE'
            'ZAVD|wDn}MS~;=e^$%xiR<wBi!-m0U3j*N3+N_KX|AI{h+PLb`4>?JlP{1y8wNH^~{PiAn1=$;VkwLT(91NuSxN|UV<lH}PrE)*='
            '-Z@gCjr*YOYcGhq@yvHCn26E}<JdalN?ZLqk6k`>aQx<vEwu}pocOWFw+Z}YG!HR`)`8x#5-lAJ_`YU3>gf=Moe*y}Tw3nJ!>fnq'
            'ZyQ*=Z0ka*OXGcLG~3`uS58OKmgAE35Ks-1uw3mk2UdpH2VxzaB8qOtZa+c0%f!}xPn)?+M(t@zR4vh=jgKe{mAnFtS3ySvsF(j^'
            'c&%H`91ACXH=(P*IbA(1FzOW_pFF|G6mRN0Kb8uQ{b-ZXVt6lIzK2#P_<$G=yvD8)*l56tXUmw3pVD#yVDGKMoo>ywFD%*FDhTuC'
            '8Qnr}mzaz<1%J=ERow+Ui%pm$dn#%EZY4Uuv^r-~d%@d`_adgDyx>KmgpdcyW_E)Nm*KXA^|y)(=_@j{8;2~0P2J+JaGs6m2HGLU'
            '-(4rsNTNAM^@+}9;!MD)sh~JWxtb%FK%`}a^E+LvK{LjzkKk07Gtspofqbv4T<$C%+V^{_SQIG6y<FB^40ZhleZ4UiU>GV%@UToi'
            'Lz5+0*v1Gh1~rr_?RCMz2Q(=iLCH!I$4aIwzDy>wT~?J96*6JdEas6?;^8>n?$SBMd>3wAd{leheRwnb<>m9Ux3i~jp1%6~`OEVk'
            ')#TXQo2XceKx+fzE57fde8lxGammb>06B9EryMl$X@+iXjX;u&3ap4_tzEs=_WEH}%`FDBQ^2T>Xosa)Qz8ZiAt%>a9+zwH!sTz&'
            'm8RS)I-x)m+~Pnn)kY%aIl5aiXoJU+Y%MpPnZ5)ZReRDEz;G$&(K@`9@Kn5atq{TSM@?u2R!`*uu~^9!ufoL&qgk&mqxRt?6?6p^'
            'jPEDl7<?QraKDrb<8$FvI0&iiYoANMVL1TIQ6B#mB_@B*8k3e_*{F;q!VYeay6VxsqJ(w>Dsa@E2u%^<;yV2sXQSU>PK{j1`~_Lh'
            'Ur`FFd9nI~rk++w$t|4LS`o;dICX|oK1X`H0ZmJ-!%q}(+w!%XP?RoyK*ll$cv+bHXR0k7US-KNX|J|)Ky@Y0^1i3alI_L*Mm<d>'
            'Nyp0EiTUd5Rak}z=o(nKBVLSXTRw&@Hj-Q(DQQakbRBB`VP6qH(s_BnMu;npP-<n>SPlx4yfZmI!YG*d8dX?<rEB~WbA%=!?gFpj'
            'fOQ_VR2rtzCh8^KHNqE!m;%27%Bb#xwoIp~oY{1xOI`L*@pqQBPBTfEZ^SGuia47MIXz%{gPBi^ky~W0a5RNUc3nj%tW&{XjIkc?'
            't7%8J3a41h;fY$?9KBOVAO8OI-Ahr}Z)i<1aDADj%SsJyva|xJ50F-D<5H2c1gnz3f$-P5az$~I<p~$CJ@Pj7Q4ua9H1=RoC&0`<'
            'a1VY9+<l#5-ej@C$`R>~%5cX!m=hPZT!c42N<ju_eIv+iLT4=g&m*j31nAo#{?sNaQw~($A)xV5>u%?xmzaYdOkCn9*xX(n&JjBb'
            'X4zbCZ^f*G;Qtntw44GeP6;9B2WP0!5*<)eQkvb%niBGsRlfu0ri{qBO%PU$DUnhhABg%dt_k+FwZQh*0Bf91!SL|XUcI{ds^M@('
            '-UG6XwjvvYAOYxGfFe^oKAseY<>{LN5rhDvG$kO9XCiWGktzC!webrp+nf^?`zWhn$}#2#tus1Ej8qnvg}K^_3Nw*nqb&*`3sjV9'
            'UGpUpM$}u(F}g!Lai~rvn&%sRg>$v3de{?Uj~l!Ji9#wkbDJ`)Si(aw4Q4H)7=J|}Y@>&R+{7LQvi5*h_(U$gL1SB|!p6zK4I22#'
            'c45{!r_cr$FKM=V+)={DGSs(Du5o{St+QgZT(hH*H7307XV40Eq>lI9A>?#|*tN{H&B+B;`QI0~1CT)ZN`++}ukcm{R+>UA@U<O>'
            'bkVC4=C%VRhHg`!(+iC~p;Jy+Mz2-?Y3<x>1H`xjNtpIPA-5M4B5<&c+XCd>ByETrNdttlic-q!+sDjh3Pjf(>&;iaKGFN$LcDc5'
            'u8&H)Mg7uh(Mh(_`q(axLwAz?Q5cF_U1?9Dz^^Y5*l8x_T}}UQn_{Lm<ybX++N}Gqe`Z*48}tmk1TA;wKW#^gEv@RikK25mEhW49'
            '2bAOD10ARVm~vc%aA@s9lxsjL$800YDGB=js+Fkz6{}GF3l^Mum_&b%;#1b^t`bxaD#X)0KA3-jDpYviMSdNXBa$~p`l!XiH2aU<'
            'E_xyXs=taqMoYOwW@Oj`LJhvCycP4{=5h8=oh!Sw#q(R5ISx&`nZ@{9n*APCv3}<&B%QU6zq%sHL3NJ*#j2Ay*&r>^zFysl?ktXw'
            '-l4Tq7aWjUsSHS`XbgCWvGf|SpmL5zVY=NSOn0`qg3S^GnPP3V2-rjcl({T?L4r;4C<>DrkVh*}80gNdJX=I;MhO822_`j>n3i;B'
            'lA!i_6YV(ONa7-#G!EO_=mr;JGl@_oEbh&dz>yYkI_=Od4r&HvMVzXm&j98A$PE)l9o}RiIsJDeznF6fmxzKt?aI5jK_{Kjen(aZ'
            'M`Uy%iGspVo!}i;Cf9Kw`sk&YCBiEQmJZe7(OZ--I5atej$6|zf=%9u<;V8R4E1E2C1@KtFYb^}+bXX+nK1l$d-+&@nJr{qe1^f^'
            'Up5nBs6eTc7UKp|{y*-AZ+!'
        ),
    },
    'approval': {
        'sha256': '8e6a6e6d92d0b51fe6ce55d06c96ec76680530d63e2335bf2d6961293a8f8357',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;NTXWmSl78o}m;*lutfF{ocPpt=bm>$n5}k0PNM1=!)~j-nATT5mf&dp6%d|@WeZTIW0R|u`Ih)=1!IA(>qo*%l-v*xNT?A>C'
            '#$k}CZB~@0OP%UGDB~<ut7;RZr@?lcXP<F<8I*e1>;3lAM|J<x<WBu^bNS(Ftj7O2xx2sXogVzxJGZMhRZ*(BR)M-3zdye{zaOij'
            ')LT`qf>N#U5}g+6{N4Td7MHZD3Z3I}p@Oiif+X3gLhH0gw+x3mQbAtE3p_AX?}MU<)1{gfVIFVGsW7dhv-`Z#GZm+0#$*>&8uH%g'
            'wLMFz;Cr%EZFG>z1xsGV1$YfMTfEY<lsyoYWx+-(h%f2!R>E8}Z*i$kPqE=emV;8d6KSe2tC9#C`>Yinq{oKpp4Q<VX;$J%Wd+_N'
            'fe(lf?gc8Z=5xq|8G#z^WJ8X+bXw*+C=r`2h&d~Z=8@rcHyfx~Sr+sg-WBy`xlSXUPpdLc<Z7goc+O&h(1S44kjeE;%@w4wtTuQT'
            'Eyuo7m8E!4f7W?VZ@?@levb9yG}r%K#kr1V%Fl8&TLf`JfBap|;<Tt1i#UuyWf}!#fUCV(1QJQMCDPMv5`=n{CE#q<hj#O9lZo0g'
            'HM1I+)n=o=gh^GzpNZp@&h?;2yfI&oD=lgw4Uzb&?KX*x#_xVQKmFJ5zE|0T|I}Yd*<qHJv^{J^cA_JQ`-jQ(<>dNDb^iYS?ai;}'
            'SL*&oUEEyXkFW2Kn7-Km7;@P)!pn7-<x$aWVrpJDEmi?ofqICOC|+u~i{Z<%(VOqcR()frrT$VvoIw@EnOZ=)_~UaNY54<Un}sVK'
            'uBZ0pet!m1r;sTmRfx5k7cC&*mdT$KDy;IH<X{*as3OC1YfSL9z<A7{2da6MrgZWC{%)Y4Z@R>mMHh*(a28Z$w!!j{T{=#|RvgtH'
            'xTT$DRne>E1yd=>O(2N^n@UTp3WGe~5m{&m-t`#dq@e4pT+v0+zpA!L7DUBBZL4__7b{&bQ?wOqj=U+=<ztqwfhzguI0UYFp4VeB'
            'Os9*gta3e_s(7=_ax!#yCOc5k>zQkSrWO3e{`q^6rLvrMQ^tS<bI1M-#p3_3S*qo^ZBP;c``CN@+3OkW`r&Fk{TT*#r_Pk`K@_28'
            'cq6|^y@B#X@buf~4BW$za{5Cg?)7`UHvkoqr(LONrQpcr_+oN5xw)=&JrLjl@I`S@Umd}90=mq&1%hIQ1rcEr)nQzaOU7lPlZEm('
            'S)|EGsyxKcO-7DfusLVs0qHWq<*X{VRS9aqL9!{~m=v>!mwmP|(5rorq|7f2)^KCmrkc@C9z6Sc@6&7ua^hx-SOZ)Ih8btW72d05'
            'Cv0Hx7U-6O`wS+>At`|~2$bWQd6I=|;5H}%vc;glask^lI2>t_J*^^;VL~Lo#+F?SANHWoi}BqZuw(lE>il|Y#XtRd{1NJHHbP2U'
            '!hJ+bi(Eroc{T^`h04K3H4h5j17I;r;&3-2&IpH7!eB%kV&0$&89jcBDh_&6ia0w`w}KCrc-W}eVB;1+R8!~}_O8djj&G-qInpna'
            '%keFl<de5V2=ShY<8!ORO|TQhr5JBt&*)1E1x!wE$MDOd1QQ!lOf2?j9wjA8t{~7Na51mq#Gn&4Sfw>#1A(N0z?IIa6FES!_73&P'
            '_K*Y>Gh42*Lc8?h6lPgSo`5$wQNH#Pkb7Lk;Y!W1gLNlEAcM_06A?>HDJftmvUv}%tswQCUtcOpO7JdrDvrMJuy_F=e(<c}c%L3T'
            '5RGDn9q-dy?DLli+mSJI_8<tbQ161I0D|3)|KkIMd^x=xUqe$<gs02NJxq;Zq^I!DD|NWyWN&J$mCS<m9(e<Kq?19}%e5$9xI^xW'
            'y_ASM#nfU{4{&sIJNa>PeSS4P|8P0E`I8I^v`pazTFlyE2=D4edVzE#NGkZ^)CibTDZhcXC=ScMI{hdAG7_?$H#Un3&SrL`vRVZc'
            'X^gRofeK*+9$I{>5iyX6Je^eOMwcfe^*|O-d>W{eMvJ)oG#n0}aZ$3NY3tqt?tOY5GQo4CbDM3sQ-&a5swge=-dtaOq&%=>;1Z(@'
            'Tt9SN&@lJ|qNBlPm<-I)f)$y5q_?z|f-52>yT<|37BMMxYLgeBTf_g%+&on}06RAMQ;Q^6);_=q79fJ`-%xw)pTCJEfSvtQ?0jHw'
            'u5r=+0mk;^J0sQ(`67$Cp&$Wj*cjhXjE~ooTsU6TR*H=lRl<Snl(Q{tncV&xcZW=B#qtK`lsUNLfI}!Mc^@T<Mu^r(`6nC2h>hX_'
            'i24cs^YBSD-EdA$RaZz$UHYDE&g)ZHczW*3#b$xM!F1w<LrR`BZrHcA3aN2a@058Oaz-BcFZyv`IJL@}O4L15ZDwmEbh&C2O3$|l'
            '$SMaj-E}tbPbo{;7mVZ|9l^BLP<oXj@g$c4sD#1R*qycXX^S<#7RzhUS3d}Wceq4$kmd0*PO(MK@~1^LFXJ-Ng;y`r@CkJs`uREG'
            'ln{*L5<ues*CUsc<}%BImbKK*k(Y@{?qe*nW%ezDU<#u5J>NW{%Dv}|#K9iFw4mZ3NJH)KJvm_4=r{blQ5T%WOQxA`j4KPK3eqAP'
            'P;OFEL}k`3(gbdn;>c*9!Knc&?^~tv0twkGX}4MLYH)95`!yt_m(}x<igJ!;>ZxAf@sr-jNT#{&Xh*`T-(2ZC&+RL7w7&)C`2z;('
            '3E=q$Zn<x2yX~m$RPx-j$r7aIo>C$0(|;pG)tvJJ{H)W0cU6)oAITPK+>rdgUpEd``ioPm2eb^jL-G9K{@v+66xZuR3))7grg9w0'
            '%YA+^;g09^hdK?xr0-Sb;`AS0zdytagudTz^Bmf)-IPjCKb!x)$G6knpf1h)cg=PE#rgHk_2lCGO8xsCsz81vg&n^@9g1MH*1Ljh'
            'gkp<Sy6Bq>qG=NrnG{P1d8nRa!24HQ<Z~~qt_*|ZG1wJqN9k%&-)Cw@Xt`dZdIz2r#3kOWDB$HJ4x}{@9<e}?H={WD7#Gw(<We@*'
            'RN?DL5QPixxLBLAj|@71M5X+rBtDH=1sdcOC&?*aaB4L-#EL2+U=s;=7*(5X(V#k{Ilv<Y^$CI^jN>ztoRAQwBtq^y;%~hT5Hzy9'
            'IP<*$MUjz*Iv1J@kT;w`Ut*Gbx_`lB3~Mh~2Q`3v>M;BnV<sZ7m$ng`2hO@Y3BQjq(##BK!Y`+e#2i5t018Dw&21zLS5>+uaDt+9'
            'KM6MTC>WV1hg7Qi-+%YrzkLsB&EXqu@DetL)t2<>Go_Zq%>8`<*>)S3D&WFjBmZT;AE*QMfh#(j<R9o9+J1%UmT*w{vW36FfXd9Y'
            'PH59it6WjxYx8Snz-+G5>fx9K@^#BbGNqSWEd$$4F*Hr7T>mpP-6o>6aLn^f&D?t4>AG{n@!jNVEVg0Rnfjyqo&}{Zrkr=yNZAy`'
            ')NuF}WtSf3IeWcn*EZz`lCGG0EJlpMJZS{J)1~tDa4EhUkN^?PPdSA)I|PKRgQX@jLfTdVLP{>oC7a<E4;B3q;oBI1AWGV?LA2YQ'
            'YfLhyjX3-87v4NZ(Anr6_Jh5^-PaIyl~<?%iRRz(XXVfJI)PYIT<g+)?%OokGC}EJw-wlMP=jtKjee>9)OWL)PD*9z4U8{P=7A^d'
            '<+eBOZGSJ(=yE<JP*rhA#iu&4R8PpjmzCOO>$fO_f*qa=!eFiDG0=RUXj~*30cQ<_&0_^D2C8o~GN@tu;k}L87fPbZi9-L+ZppBb'
            'Mu9pZiYLJS(C|aML8yect*L(pm*ATST6!5LQJf!T*x=dWsBgD%z*0>2rP-8e>#)CfOD*S~w<jy-<{uW3^k|T(36Yl8U521=<R6IV'
            'p*AiIw_QV>{kMno-Zc#XRvxdv$pG5&<jl`u<bWWX279x6DaI(o?N_b!t$j&4%Zxiw*WYh0_lI#IvHLY^TtKuie&oM7Wn5XYgH@?{'
            'l~l;$gD3`~G-QD_-Z2UA+Z+x|6M^8`Ztz>02#oU6W?8#bbI7?HZ!RC)_<>EE7;|vH4Yg*!2lnc`)T9K`(w}Qe2c{BZUdaktT|=7|'
            ')I$ayU8h<{JcX5nwoU>DxE3Z=WHV-Ni7_djb>PxY+EhARWxxuAdnY=nYpzH)Rawu>YUR{HD_vyNq;D!B+6)RI`r*_`cHxvdi4|2>'
            '9Z-<V{dTeE%qp?s8nD(X+H8QvJxPe0o1D9l8+BLl4MNdYsrT^7)@qzG<vXW8fgL_{nV}xo!Q`&dV827bVV73dT}l0ghu%(F67Y<<'
            '%YoeyErzN1cx2Db=*gErMMfAc!E=w9=Xhj4$0sqPm$$?6e}UsRTmw*oWfJTF&g$f;#n6dm=%oLAqPAHA9jyZ-gs9$Ph$ac@SiK3p'
            'cJSKskIeP^4n+6&1Y{<eZUg7O9wJkVn+rJV<sz&3IKJe9!lc=L+cKzAjw-a!GQtiVlBPJ*%A2T19$V0X49_%Pd8mdrtjQQY$AP(Q'
            'M+)*b$35NvkNDUC`!Atf(`u%)*|w25fkk<n=w`@R560rcq1Ozkn#L1NGN8Q|2OO=)Lw6ob|K?MtN%XqibZn>Nru01bWLhujA6tFU'
            '^4Ln=QHxaFT%Uhb<Le*CS636N`q?B?sGCojK<n$QK>dm=YjhwjSa?q-19dX|J&RLe$P7<D^`D38Rze4nfzhU%I#j5gkgf$WrRJMM'
            'nFbQJ6q&_mt#%diyZAI%qi(3#P$f)$10Ex4(}EOS8hWMN|8PpzLIsXKM#EF9yUvtqpqx--4}x?KjmX_2BZvB+$$V$1z%q5gztjBL'
            '+mC2++(y$W6~oJnf{+A+vlC14gbz5u<ar(yOZ_-I!sAStx!^0uOl;-eo~s3b!QYrk<<oi|*V=xxM8l{*JWx+k7b@Ku(J!hZNCX5-'
            '@3A?U;||aj5=c35z(YS(R3Rn$iz=Zs-mpe+#38#r4+VBm`$yF5*-6JqHv0&`({e6r<Q_$0n4ptjOfShY8XlP(!a$kiTFRFh-!|u?'
            'UCjZZKB*;}mpn@^hw7Ykln6a=zN|FOTk2C3x&EEAakGPWc!Z4W>_jl+gef1M)HAO^B!16lcbpj`Q}YY`(i{TX`S$UC&*@vR298Yl'
            'on8fR4+^F;4^^4R%T*~8yFjw5_Wb`3P5%GNo<KWqc769aUC((;ea-SJO+ZnAZXIvHBm_hNg$i{SQQ!_YI*EtvcOAo)-nW%|-jVf|'
            'p`p4-Lvq|ubIrc0LPW+mjmaC}ZZVYIO&yk5Q8J#Ub-xYuG94+S)IUd9P&*_Xj6wq`0<lRv5b8Bp*e91q)v`*flm!vD%WhNoyipfs'
            'jQhj*-3=U}4$(YllQe^Ds2E5iTw7q_c}vkr>fgw~#GR)xO4TRT5)A+-99TGUT7wmgvZAbH_<Dl$f0-A2LuJpQ8lHYRzMNcLPDZx<'
            'CuT<_J|~FlB(9@n9nPbebap6!y8+)WgLAZkIb!*NKyAXi1a@w`#La0hM0r;G;b?xe!EhkH+eB9){Gp|H(#43sBD~Nd@j6{@^jeXf'
            '@3Oq$`Ne)N>G#!|ww&&z4l?_eNXgaBHxxT$e7OWfadEcdCfTzlb;NOpG|Srh{b7<l>Kv^7?X1Otz5cBLvDh~m0Rxt;nsXbBecg0('
            '=p`qy@}G_{_%QmH?_mtXVZWn>g=fJPMtJw({QBpK1&191vH#?5!=XYAm108*8mfu#ARI>VAvh_rGzS3b6rn7&y(gGcmWXw{Jp^{J'
            'z#N`nF^^3V6zpdA)ZhPquO0ue*8WVL*b$*yHXf@JNuD`;dJU0vipHLs-@+q!{M^Pmd-uV%GpPP(V*J5p=$K?dCrk0@!jpkc{gz-s'
            '=O#^hhIsqr1mEW_JzhI)j`X$LGBp<?Zaob4?|>$?WtzVZJrmGa-kvfZrrD!!4;l}vGNjI026RC7?YpfUE&{NSe^QdwY04ZqpN<u-'
            '_qZ(@Q`*!yoSC<^kDwndZVvU;>{0c-lA%>9$ae_Y2BkbfU8&5c9&*>PW)4yXWs~u<xVp#p>e>c{Bu|Iw7qN$vXX`Ze$0b}UxA7ml'
            'FsDYW#rV#YshL6dyR>9n%(wY>p;dEh`W+CLNfz2=(@WG0m!M_^XXfInAcU&l@s<dG+rhQwe14(s?$58rtOU33!Kc~C3XWGAb&1gw'
            'Va<^sI-_`gb$3Ij3~WvlQOYGQ)|{|b%Z$$~%PgPghcpj#XPhp#Nnre!tF6;hY50BSsbf|xSI)Nz^0fv2-V{0^GZe)d;kBy*IR?mQ'
            'qaC+4X9DDio%6Q=m=TuOhwTnGUn%N36rJuOz3;(mK~-H-Fs{ZjTtZ<cb$bV_<?2ZvMMvN3b}fZ`ujjt1=AzXrtzBzxPS63r_PpR2'
            'ymU$uq5a~vJ7vgzlLFC29eJJ90Wxl!SJGnYgeXZIw3?gD6cK(v6%-19_$y-J4naSlVybs^*5nGa)yK9z{Edz4*W1zut)i%cbRBoH'
            's+BH+t%P14o`B5ri&j8q0)<^SA+#DgkPN=2DU(_JIKDtSGWoBOWssB*!M`QFA`>n)drA7j^KI^so;_JE>Rb*u!J9s_sa})i4b?}J'
            'v6<`>Wv3zaNA&nc{pw&um0Cmvr`I?4s`<Xnw2%b%Ms3{V)`}s=*jYEVn{)%P+8j1^&h|Rb)2Z6Ox<GtQyG@PQC|FQcG^!<2E}-GS'
            'EghZKD)F>_sY;|wz1t-|UNjyX2)}}Q0^J?R+uJ;6vdZ-|0~s-8_zV|G+&Tql-^EGN!D6tRD!thJZSLAO?+bza)dujb`AV^Lz5r9d'
            'o!?*lG`^J6OTy|v{WiJ(>E^?|^JPR_)(uiV1%*1Nu$>ZaY8av>g_ch^#d8SrlMHrUp8~S3BK5ERW=PE58IGt4*5q_vkh-;Ys55KH'
            'b7l@t(MPXxV8;2(mA~R5$K`&Dnn@IyQOKIL;~0Mj5Y*Emd4+Ts`I{n1IsV=>CMl6|KV0Hr)ZRC;NBiAxZ4K|he%$^nRQkL1hiKeq'
            'uTwd3Is=0VeWJw&CHH!otrp<R?MCOZrrl_3-MZ(v3%2Fap5tq+42xdf5cqPhQq?oSzo45jxxN@*jei-dpU*#DPFfYR)63uu`cE_+'
            'KEhf5g+7J>6w%?)z6A{uo!-S>g<RXo!3x>leehcD(k)i)WBbLbV-DWPwV^}L{O^j@BceN4jJ{UgKAV6mE>3-athYmrPQ$gvYV-ZI'
            'pfw#lo&rXuT-%zHuI)m$thR}M5ZtE!Kh@BBk)<d$kQsz}#FWU{`5<=P^=(cwQ}h=eWGd`8pX_$(YBGerOtzxzF*OOs7Glvn+LGA_'
            'm&WpqgN#^cTI77$Ur2^gA0^p8&g%{81LJ4*r1GKjz4d0y@oCzrFAuc_p+`5=+#t#Jd;bGEZ&}L'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
render_utils = _local_pipeline_modules['render_utils']
video_analysis = _local_pipeline_modules['video_analysis']
approval = _local_pipeline_modules['approval']



In [ ]:
ffmpeg_bin, ffprobe_bin = render_utils.resolve_ffmpeg_binaries()
print('FFmpeg:', ffmpeg_bin)
print('System ffprobe:', ffprobe_bin or 'bulunamadi (probe_media() ffmpeg fallback kullanacak)')


def find_first_existing(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None


## 1. Girdi dosyalarını yükle: `hot_moments.json`, `selected_script.json`, `selected_voiceover.json`

In [ ]:
def load_json_artifact(filename, override_path=''):
    candidates = [
        override_path,
        str(RESULTS_DIR / filename),
        f'results/{filename}',
        f'../results/{filename}',
        f'/content/results/{filename}',
    ]
    path = find_first_existing(candidates)
    if path is None:
        raise FileNotFoundError(
            f'{filename} bulunamadi. Aranan yollar: ' + ', '.join(c for c in candidates if c)
        )
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f), path

hot_moments_data, hot_moments_path = load_json_artifact('hot_moments.json')
selected_script, selected_script_path = load_json_artifact('selected_script.json')
selected_voiceover, selected_voiceover_path = load_json_artifact('selected_voiceover.json')

# Kanonik sema dogrulamasi (Notebook 01/02 ve master notebook ile ayni fonksiyon).
schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
if schema_errors:
    raise ValueError(f'hot_moments.json semasi gecersiz ({hot_moments_path}):\n- ' + '\n- '.join(schema_errors))

print('hot_moments.json      :', hot_moments_path, '| status:', hot_moments_data.get('status'))
print('selected_script.json  :', selected_script_path, '| status:', selected_script.get('status'))
print('selected_voiceover.json:', selected_voiceover_path, '| status:', selected_voiceover.get('status'))

if selected_script.get('status') != 'approved':
    print(f"UYARI: selected_script.json status='{selected_script.get('status')}' (approved degil)")
if selected_voiceover.get('status') != 'approved':
    print(f"UYARI: selected_voiceover.json status='{selected_voiceover.get('status')}' (approved degil)")
print('Not: onaylanmamis girdilerle render yine calisir (inceleme icin) ama nihai durum')
print('     review_required/failed kalir, ASLA approved/production_ready olmaz.')


## 2. Render yapılandırması ve onay bayrağı

Tüm render mantığı tek bir kanonik çağrıda (`render_utils.run_render_pipeline`) toplanmıştır: aralık kesme, altyazı hizalama (gerçek seslendirme sesinden `faster-whisper` ile; başarısız olursa orantılı fallback), dikey kadraj + altyazı yakma + ses miksleme (tek `ffmpeg` geçişi), teknik doğrulama ve onay kapısı değerlendirmesi.

In [ ]:
RENDER_MODE = 'blurred_background'  # 'blurred_background' veya 'center_crop'
TARGET_WIDTH = 1080
TARGET_HEIGHT = 1920
PREVIEW_MODE = True  # True = hizli/dusuk kalite onizleme; False = uretim kalitesi

DUCKING_ENABLED = True
ORIGINAL_AUDIO_BASE_VOLUME = 0.35
NARRATION_VOLUME = 1.0
DUCK_RATIO = 8

SUBTITLE_FONTSIZE = 20
SUBTITLE_MARGIN_V = 90
SUBTITLE_MAX_CHARS_PER_CUE = 42

SOURCE_VIDEO_PATH_OVERRIDE = ''  # zorla belirli bir ham kaynak video vermek istersen buraya yaz
print('Render modu:', RENDER_MODE, '| onizleme mi:', PREVIEW_MODE)
print("NOT: onay bu hucrede DEGIL, bolum 5'te (videoyu izledikten sonra) verilir.")


## 3. Render çalıştır (kanonik `render_utils.run_render_pipeline`)

In [ ]:
render_manifest = render_utils.run_render_pipeline(
    hot_moments_data, selected_script, selected_voiceover,
    hot_moments_path, selected_script_path, selected_voiceover_path,
    render_dir=RENDER_DIR,
    render_mode=RENDER_MODE, target_width=TARGET_WIDTH, target_height=TARGET_HEIGHT,
    preview_mode=PREVIEW_MODE, ducking_enabled=DUCKING_ENABLED,
    original_audio_base_volume=ORIGINAL_AUDIO_BASE_VOLUME, narration_volume=NARRATION_VOLUME,
    duck_ratio=DUCK_RATIO, subtitle_fontsize=SUBTITLE_FONTSIZE,
    subtitle_margin_v=SUBTITLE_MARGIN_V, subtitle_max_chars_per_cue=SUBTITLE_MAX_CHARS_PER_CUE,
    source_video_path_override=SOURCE_VIDEO_PATH_OVERRIDE,
)

print('Kaynak cozumlemesi:', render_manifest['inputs']['source_description'])
print('Altyazi kaynagi:', render_manifest['subtitles']['source'],
      f"({render_manifest['subtitles']['cue_count']} satir)")
print('Render dosyasi:', render_manifest['output']['final_video_path'],
      f"({render_manifest['output']['render_latency_sec']}s)")
print()
print('Teknik dogrulama (passed):', render_manifest['output']['validation']['passed'],
      '| probe backend:', render_manifest['output']['validation']['details']['probe_backend'])
print(json.dumps(render_manifest['output']['validation']['checks'], indent=2))
print()
print('Onay oncesi kosullar:', json.dumps(render_manifest['approval']['conditions'], indent=2))
if render_manifest['approval']['failed_conditions']:
    print('Saglanmayan kosul(lar):', render_manifest['approval']['failed_conditions'])
print('Render durumu:', render_manifest['status'], '-', render_manifest['status_reason'])


## 5. Render'ı izle ve ONAYLA (ayrı adım)

Videoyu **izledikten sonra** aşağıda yazdırılan SHA-256'yı `RENDER_REVIEWED_SHA256` alanına yapıştır.

Render onayı yalnızca **tüm** koşullar sağlandığında verilebilir: teknik doğrulama geçmiş **ve**
script onaylı **ve** seslendirme onaylı **ve** seslendirme-script hash eşleşmesi sağlanmış olmalı.
Herhangi biri eksikse onay reddedilir ve artefakt `review_required` kalır.

Sadece izlediğini kaydetmek istersen `RECORD_RENDER_REVIEW_ONLY = True` yap — bu `status`'ü değiştirmez.

In [ ]:
video_path = render_manifest['output']['final_video_path']
try:
    from IPython.display import Video, display as _display
    _display(Video(video_path, embed=True, width=320))
except Exception as exc:
    print('Video oynatici gosterilemedi:', exc)
    print('Dosyayi elle izle:', video_path)

render_review_info = approval.describe_for_review('render', render_manifest)
print('Icerik SHA-256:', render_review_info['content_sha256'])
if not render_review_info['reviewable']:
    print()
    print('!!! FINAL ONAY ENGELLI !!!')
    print(render_review_info['blocker'])
    for condition in render_review_info.get('blocking_conditions', []):
        print('  -', condition)

RENDER_REVIEWED_SHA256 = ''   # yukaridaki SHA-256'yi buraya yapistir
RENDER_APPROVED_BY = ''
RENDER_APPROVAL_NOTES = ''
RECORD_RENDER_REVIEW_ONLY = False  # True: sadece "izledim" kaydi, status DEGISMEZ

if not RENDER_REVIEWED_SHA256 or not RENDER_APPROVED_BY:
    print('Onay ATLANDI: RENDER_REVIEWED_SHA256 ve RENDER_APPROVED_BY doldurulmali.')
    print('status olarak kaliyor:', render_manifest['status'])
elif RECORD_RENDER_REVIEW_ONLY:
    try:
        render_manifest = approval.record_render_review(
            render_manifest, RENDER_REVIEWED_SHA256, RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Inceleme kaydedildi (render_review). status DEGISMEDI:', render_manifest['status'])
    except ValueError as exc:
        print('INCELEME KAYDI REDDEDILDI:', exc)
else:
    try:
        render_manifest = approval.approve_artifact(
            render_manifest, 'render', RENDER_REVIEWED_SHA256,
            RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Render ONAYLANDI (status:', render_manifest['status'], ')')
        print('NOT: onay yalnizca bir kayittir; bu notebook hicbir sey yayinlamaz.')
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('status olarak kaliyor:', render_manifest['status'])
        print("Sadece izledigini kaydetmek istersen RECORD_RENDER_REVIEW_ONLY = True yap.")


## 6. Dışa aktarım: `render_manifest.json`

In [ ]:
render_manifest_path = RENDER_DIR / 'render_manifest.json'
with open(render_manifest_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest, f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_manifest_path, '(status:', render_manifest['status'], ')')

render_validation_path = RENDER_DIR / 'render_validation.json'
with open(render_validation_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest['output']['validation'], f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_validation_path)

print('HATIRLATMA: bu notebook hicbir yere yukleme yapmadi (not_uploaded='
      f"{render_manifest['not_uploaded']}).")
print('Yayin, ayri ve elle onayli bir adimdir.')
